## 1. Connected Drive to Google Colab

In [ ]:
# from google.colab import drive, auth
# from google.auth import default
# import gspread
# import pandas as pd
# import os

# # Connect ke Drive
# drive.mount('/content/drive')
# folder_path = '/content/drive/Shareddrives/CV Screening Automation/01_input/'

# # Daftar file Excel statis yang wajib ada
# required_files = [
#     'template_screening_cv.xlsx',
#     'skill_matrix.xlsx',
#     'academic_matrix.xlsx',
#     'division_keywords.xlsx'
# ]

# # Validasi ketersediaan file statis
# print("Mengecek kelengkapan file pendukung di folder 01_input...")
# all_files_exist = True

# for file_name in required_files:
#     file_path = os.path.join(folder_path, file_name)
#     try:
#         if not os.path.exists(file_path):
#             raise FileNotFoundError(f"File {file_name} belum ditemukan di folder 01_input.")
#     except FileNotFoundError as error_message:
#         print(error_message)
#         all_files_exist = False
#     except Exception as e:
#         print(f"Terjadi kesalahan saat mengecek file {file_name}: {e}")
#         all_files_exist = False

# # Proses import data responses
# if not all_files_exist:
#     print("PROSES TERHENTI: Mohon lengkapi file yang kurang di atas.")
# else:
#     print("\nFile pendukung lengkap. Melanjutkan akses ke Google Sheets...")

#     try:
#         # Autentikasi untuk membaca file cloud
#         auth.authenticate_user()
#         creds, _ = default()
#         gc = gspread.authorize(creds)

#         link_spreadsheet = 'https://docs.google.com/spreadsheets/d/1SgAhlAQHbPuQiaQ6WfxHunByHQdoR4Kx0bOMwFMjYGg/edit?usp=sharing'

#         # Membuka file Google Sheet menggunakan URL
#         worksheet = gc.open_by_url(link_spreadsheet).sheet1
#         rows = worksheet.get_all_values()

#         # Mengubah data mentah menjadi DataFrame Pandas
#         df_responses = pd.DataFrame.from_records(rows[1:], columns=rows[0])
#         print("\nData dari Spreadsheet berhasil diimport menggunakan link")

#     except Exception as e:
#         print(f"Gagal mengimport data dari Google Sheet. Error: {e}")

## 2. Cleaning Data Form Responses

In [ ]:
# # Menghapus spasi berlebih pada nama kolom awal
# df_responses.columns = df_responses.columns.str.strip()

# # Mengubah nama kolom panjang menjadi nama sederhana sesuai tabel panduan
# rename_mapping = {
#     'Nama Lengkap': 'nama_lengkap',
#     'Email Aktif': 'email_aktif',
#     'Nomor WhatsApp Aktif': 'nomor_whatsapp_aktif',
#     'Pendidikan': 'pendidikan',
#     'Jurusan dan Angkatan': 'jurusan_dan_angkatan',
#     'Asal Instansi': 'asal_instansi',
#     'Divisi Pertama': 'pilihan_divisi_pertama',
#     'Posisi Divisi Pertama': 'posisi_divisi_pertama',
#     'Divisi Kedua': 'pilihan_divisi_kedua',
#     'Posisi Divisi Kedua': 'posisi_divisi_kedua',
#     'CV Terbaru (PDF)': 'link_cv',
#     'Portfolio': 'link_portofolio',
#     'Unggah Seluruh Bukti Persyaratan': 'unggah_bukti',

#     # Opsional
#     'Email Address': 'email',
#     'Apakah kamu sudah membaca dan memahami Guidebook yang tersedia?': 'sudah_baca_guidebook?',
#     'Link Akun Instagram': 'link_instagram',
#     'Jenis Kelamin': 'jenis_kelamin',
#     'Domisili': 'domisili',
#     'Tanggal Lahir': 'tanggal_lahir'
# }
# df_responses.rename(columns=rename_mapping, inplace=True)

# # Membersihkan teks nama divisi (menghapus spasi berlebih dan merapikan huruf kapital)
# if 'pilihan_divisi_pertama' in df_responses.columns:
#     df_responses['pilihan_divisi_pertama'] = df_responses['pilihan_divisi_pertama'].fillna('').astype(str).str.strip().str.title()

# if 'pilihan_divisi_kedua' in df_responses.columns:
#     df_responses['pilihan_divisi_kedua'] = df_responses['pilihan_divisi_kedua'].fillna('').astype(str).str.strip().str.title()

# # Membersihkan link CV dan portofolio (menghapus spasi)
# if 'link_cv' in df_responses.columns:
#     df_responses['link_cv'] = df_responses['link_cv'].fillna('').astype(str).str.strip()

# if 'link_portofolio' in df_responses.columns:
#     df_responses['link_portofolio'] = df_responses['link_portofolio'].fillna('').astype(str).str.strip()

# # Menampilkan hasil cleaning
# print("Proses Cleaning Selesai!")
# df_responses.head()

In [ ]:
# df_responses.info()

## 3. Administration Check

In [ ]:
# # Daftar kolom wajib beserta flag jika kosong
# required_fields = {
#     'nama_lengkap': 'MISSING_NAME',
#     'email_aktif': 'MISSING_EMAIL',
#     'nomor_whatsapp_aktif': 'MISSING_PHONE',
#     'pendidikan': 'MISSING_EDUCATION',
#     'jurusan_dan_angkatan': 'MISSING_MAJOR',
#     'asal_instansi': 'MISSING_INSTITUTION',
#     'pilihan_divisi_pertama': 'MISSING_DIVISION_1',
#     'posisi_divisi_pertama': 'MISSING_POSITION_1',
#     'pilihan_divisi_kedua': 'MISSING_DIVISION_2',
#     'posisi_divisi_kedua': 'MISSING_POSITION_2',
#     'link_cv': 'MISSING_CV',
#     'link_portofolio': 'MISSING_PORTFOLIO'
# }

# # Membuat satu kolom hasil
# df_responses['status_administrasi'] = ''

# # Loop setiap kandidat
# for index, row in df_responses.iterrows():

#     missing_flags = []

#     for column, flag in required_fields.items():

#         value = row.get(column)

#         # Jika kosong, simpan flag
#         if pd.isna(value) or str(value).strip() == '':
#             missing_flags.append(flag)

#     # Menentukan status
#     if len(missing_flags) == 0:
#         df_responses.at[index, 'status_administrasi'] = 'ADMINISTRASI LENGKAP'
#     else:
#         df_responses.at[index, 'status_administrasi'] = ', '.join(missing_flags)

# print("Pengecekan kelengkapan administrasi selesai!\n")

# # Ringkasan hasil
# print(df_responses['status_administrasi'].value_counts())

# # Preview
# df_responses[['nama_lengkap', 'status_administrasi']].head()

In [ ]:
# df_responses.head()

In [ ]:
# # Cek data terbaru
# df_responses.tail()

## 4. Validasi Link CV

In [ ]:
# import re

# print("Memulai validasi link CV...\n")

# # Membuat kolom hasil
# df_responses['status_link_cv'] = ''

# # Pola link Google Drive File
# pattern = r'^https://drive\.google\.com/file/d/[^/]+'

# # Loop setiap kandidat
# for index, row in df_responses.iterrows():

#     link_cv = str(row.get('link_cv', '')).strip()

#     # Kolom kosong
#     if link_cv == '':
#         df_responses.at[index, 'status_link_cv'] = 'MISSING_CV'

#     # Link Google Drive valid
#     elif re.match(pattern, link_cv):
#         df_responses.at[index, 'status_link_cv'] = 'LINK_CV_VALID'

#     # Format link tidak sesuai
#     else:
#         df_responses.at[index, 'status_link_cv'] = 'INVALID_CV_LINK'

# print("Validasi link CV selesai!\n")

# # Ringkasan hasil
# print(df_responses['status_link_cv'].value_counts())

# # Preview
# df_responses[['nama_lengkap', 'link_cv', 'status_link_cv']].head()

In [ ]:
# df_responses.head()

### 5. Sistem Cek Portofolio

In [ ]:
# # 5. Pengecekan dan Scoring Portofolio
# print("Memulai evaluasi portofolio berdasarkan divisi...\n")

# # Mendefinisikan divisi yang masuk kategori Creative
# creative_divs = ['Graphic Design', 'Content Creator']

# def evaluasi_portofolio(row):
#     # Mengambil data divisi dan portofolio
#     div_1 = str(row.get('pilihan_divisi_pertama', ''))
#     div_2 = str(row.get('pilihan_divisi_kedua', ''))
#     link_porto = str(row.get('link_portofolio', '')).strip()

#     # Mengecek apakah salah satu pilihan divisi adalah Creative
#     is_creative = (div_1 in creative_divs) or (div_2 in creative_divs)
#     has_portfolio = (link_porto != '')

#     # Logic Matrix sesuai Aturan Sistem
#     if is_creative:
#         if has_portfolio:
#             # Creative + Kirim Portofolio = Perlu dicek manusia, skor dikosongkan (NaN/None)
#             return pd.Series(['MANUAL_REVIEW_NEEDED', None])
#         else:
#             # Creative + Tanpa Portofolio = Flag error, skor 0
#             return pd.Series(['MISSING_PORTFOLIO_FOR_CREATIVE', 0])
#     else:
#         if has_portfolio:
#             # Non-Creative + Kirim Portofolio = Otomatis skor 5
#             return pd.Series(['AUTO_SCORED', 5])
#         else:
#             # Non-Creative + Tanpa Portofolio = Tidak dievaluasi (skor kosong)
#             return pd.Series(['TIDAK_DIBERI_NILAI', None])

# # Mengaplikasikan fungsi dan membuat dua kolom baru sekaligus
# df_responses[['status_portofolio', 'skor_portofolio']] = df_responses.apply(evaluasi_portofolio, axis=1)

# print("Evaluasi portofolio selesai!\n")

# # Menampilkan ringkasan flag yang dihasilkan
# print(df_responses['status_portofolio'].value_counts())
# print("\n")

# # Preview hasil untuk memastikan logika berjalan benar
# print(df_responses[['nama_lengkap', 'pilihan_divisi_pertama', 'link_portofolio', 'status_portofolio', 'skor_portofolio']].head(10))

# FULL PIPELINE

In [ ]:
# BAGIAN 1 — SETUP DAN PATH

import os
from pathlib import Path
from datetime import datetime

# 1. Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')


# 2. Set direktori utama project

BASE_DIR = Path('/content/drive/Shareddrives/CV Screening Automation')


# 3. Set struktur folder


INPUT_DIR = BASE_DIR / '01_input'
OUTPUT_DIR = BASE_DIR / '02_output'
ARCHIVE_DIR = BASE_DIR / '03_archive'

# Folder untuk proses download dan ekstraksi CV
CV_DOWNLOAD_DIR = BASE_DIR / '04_temp_cv_downloads'
CV_TEXT_DIR = BASE_DIR / '05_cv_text'


# 4. Set link Google Spreadsheet Form Responses


# Ganti isi variabel ini dengan link Google Spreadsheet Form Responses terbaru.
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1SgAhlAQHbPuQiaQ6WfxHunByHQdoR4Kx0bOMwFMjYGg/edit?usp=sharing"

# Jika ingin membaca sheet tertentu, isi nama worksheet-nya.
# Jika None, sistem akan membaca worksheet pertama.
WORKSHEET_NAME = None

# Contoh:
# WORKSHEET_NAME = "Form Responses 1"


# 5. Set path file input pendukung


TEMPLATE_SCREENING_FILE = INPUT_DIR / 'template_screening_cv.xlsx'
SKILL_MATRIX_FILE = INPUT_DIR / 'skill_matrix.xlsx'
ACADEMIC_MATRIX_FILE = INPUT_DIR / 'academic_matrix.xlsx'
DIVISION_KEYWORDS_FILE = INPUT_DIR / 'division_keywords.xlsx'


# 6. Set nama file output


timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_FILE = OUTPUT_DIR / f'hasil_screening_{timestamp}.xlsx'


# 7. Buat folder jika belum ada


for folder in [
    INPUT_DIR,
    OUTPUT_DIR,
    ARCHIVE_DIR,
    CV_DOWNLOAD_DIR,
    CV_TEXT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# 8. Tampilkan konfigurasi path


print("=== KONFIGURASI PROJECT ===")
print(f"BASE_DIR              : {BASE_DIR}")
print(f"INPUT_DIR             : {INPUT_DIR}")
print(f"OUTPUT_DIR            : {OUTPUT_DIR}")
print(f"ARCHIVE_DIR           : {ARCHIVE_DIR}")
print(f"CV_DOWNLOAD_DIR       : {CV_DOWNLOAD_DIR}")
print(f"CV_TEXT_DIR           : {CV_TEXT_DIR}")

print()
print("=== GOOGLE SPREADSHEET FORM RESPONSES ===")
print(f"SPREADSHEET_URL       : {SPREADSHEET_URL}")
print(f"WORKSHEET_NAME        : {WORKSHEET_NAME}")

print()
print("=== FILE INPUT PENDUKUNG ===")
print(f"TEMPLATE_SCREENING    : {TEMPLATE_SCREENING_FILE}")
print(f"SKILL_MATRIX_FILE     : {SKILL_MATRIX_FILE}")
print(f"ACADEMIC_MATRIX_FILE  : {ACADEMIC_MATRIX_FILE}")
print(f"DIVISION_KEYWORDS     : {DIVISION_KEYWORDS_FILE}")

print()
print("=== FILE OUTPUT ===")
print(f"OUTPUT_FILE           : {OUTPUT_FILE}")

Mounted at /content/drive
=== KONFIGURASI PROJECT ===
BASE_DIR              : /content/drive/Shareddrives/CV Screening Automation
INPUT_DIR             : /content/drive/Shareddrives/CV Screening Automation/01_input
OUTPUT_DIR            : /content/drive/Shareddrives/CV Screening Automation/02_output
ARCHIVE_DIR           : /content/drive/Shareddrives/CV Screening Automation/03_archive
CV_DOWNLOAD_DIR       : /content/drive/Shareddrives/CV Screening Automation/04_temp_cv_downloads
CV_TEXT_DIR           : /content/drive/Shareddrives/CV Screening Automation/05_cv_text

=== GOOGLE SPREADSHEET FORM RESPONSES ===
SPREADSHEET_URL       : https://docs.google.com/spreadsheets/d/1SgAhlAQHbPuQiaQ6WfxHunByHQdoR4Kx0bOMwFMjYGg/edit?usp=sharing
WORKSHEET_NAME        : None

=== FILE INPUT PENDUKUNG ===
TEMPLATE_SCREENING    : /content/drive/Shareddrives/CV Screening Automation/01_input/template_screening_cv.xlsx
SKILL_MATRIX_FILE     : /content/drive/Shareddrives/CV Screening Automation/01_input/skil

In [ ]:

# VALIDASI AWAL FOLDER DAN FILE INPUT PENDUKUNG


required_files = {
    "Template Screening CV": TEMPLATE_SCREENING_FILE,
    "Skill Matrix": SKILL_MATRIX_FILE,
    "Academic Matrix": ACADEMIC_MATRIX_FILE,
    "Division Keywords": DIVISION_KEYWORDS_FILE,
}

print("=== CEK FOLDER ===")

if BASE_DIR.exists():
    print(f" Folder utama ditemukan: {BASE_DIR}")
else:
    print(f" Folder utama tidak ditemukan: {BASE_DIR}")

for folder in [
    INPUT_DIR,
    OUTPUT_DIR,
    ARCHIVE_DIR,
    CV_DOWNLOAD_DIR,
    CV_TEXT_DIR
]:
    if folder.exists():
        print(f" Folder tersedia: {folder.name}")
    else:
        print(f" Folder tidak tersedia: {folder.name}")

print()
print("=== CEK LINK GOOGLE SPREADSHEET ===")

if SPREADSHEET_URL == "" or SPREADSHEET_URL == "ISI_LINK_GOOGLE_SPREADSHEET_DI_SINI":
    print(" SPREADSHEET_URL belum diisi.")
else:
    print(" SPREADSHEET_URL sudah diisi.")

print()
print("=== CEK FILE INPUT PENDUKUNG ===")

missing_files = []

for file_label, file_path in required_files.items():
    if file_path.exists():
        print(f" {file_label}: ditemukan")
    else:
        print(f" {file_label}: belum ditemukan")
        missing_files.append(file_label)

print()

if len(missing_files) == 0:
    print(" Semua file input pendukung sudah tersedia.")
else:
    print(" Beberapa file input pendukung belum tersedia:")
    for file_label in missing_files:
        print(f"- {file_label}")
    print()
    print("Silakan upload file yang belum tersedia ke folder 01_input sebelum lanjut.")

=== CEK FOLDER ===
 Folder utama ditemukan: /content/drive/Shareddrives/CV Screening Automation
 Folder tersedia: 01_input
 Folder tersedia: 02_output
 Folder tersedia: 03_archive
 Folder tersedia: 04_temp_cv_downloads
 Folder tersedia: 05_cv_text

=== CEK LINK GOOGLE SPREADSHEET ===
 SPREADSHEET_URL sudah diisi.

=== CEK FILE INPUT PENDUKUNG ===
 Template Screening CV: ditemukan
 Skill Matrix: ditemukan
 Academic Matrix: ditemukan
 Division Keywords: ditemukan

 Semua file input pendukung sudah tersedia.


In [ ]:
# BAGIAN 2 — LOAD FILE INPUT

# 1. Install library jika belum tersedia


!pip install -q pandas openpyxl gspread


# 2. Import library


import pandas as pd
from openpyxl import load_workbook

from google.colab import auth
from google.auth import default
import gspread

In [ ]:

# FUNGSI LOAD GOOGLE SPREADSHEET


def load_form_responses_from_google_sheet(spreadsheet_url, worksheet_name=None):
    """
    Membaca Form Responses dari Google Spreadsheet link sebagai teks.
    Ini lebih aman untuk nomor WhatsApp, link, timestamp, dan data yang tidak boleh berubah format.
    """

    if spreadsheet_url == "" or spreadsheet_url == "ISI_LINK_GOOGLE_SPREADSHEET_DI_SINI":
        raise ValueError("SPREADSHEET_URL belum diisi.")

    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    spreadsheet = gc.open_by_url(spreadsheet_url)

    if worksheet_name is None:
        worksheet = spreadsheet.get_worksheet(0)
    else:
        worksheet = spreadsheet.worksheet(worksheet_name)

    values = worksheet.get_all_values()

    if len(values) == 0:
        raise ValueError("Worksheet kosong.")

    headers = values[0]
    rows = values[1:]

    df = pd.DataFrame(rows, columns=headers)

    print(" Berhasil membaca Form Responses dari Google Spreadsheet sebagai teks.")
    print(f"Spreadsheet title : {spreadsheet.title}")
    print(f"Worksheet title   : {worksheet.title}")
    print(f"Jumlah baris      : {df.shape[0]}")
    print(f"Jumlah kolom      : {df.shape[1]}")

    return df

In [ ]:

# FUNGSI BANTU — LOAD EXCEL


def load_excel_file(file_path, sheet_name=0):
    """
    Membaca file Excel dan mengembalikan DataFrame.
    Default membaca sheet pertama.
    """
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')
        print(f" Berhasil membaca file: {file_path.name}")
        print(f"   Jumlah baris : {df.shape[0]}")
        print(f"   Jumlah kolom : {df.shape[1]}")
        return df

    except Exception as e:
        print(f" Gagal membaca file: {file_path.name}")
        print(f"   Error: {e}")
        return None


def get_excel_sheet_names(file_path):
    """
    Mengambil daftar nama sheet dari file Excel.
    """
    try:
        wb = load_workbook(file_path, read_only=True)
        sheet_names = wb.sheetnames
        wb.close()
        return sheet_names

    except Exception as e:
        print(f" Gagal membaca sheet dari file: {file_path.name}")
        print(f"   Error: {e}")
        return []

In [ ]:
# ============================================================
# LOAD FORM RESPONSES DARI GOOGLE SPREADSHEET
# ============================================================

form_df = load_form_responses_from_google_sheet(
    spreadsheet_url=SPREADSHEET_URL,
    worksheet_name=WORKSHEET_NAME
)

print("✅ form_df berhasil dibuat.")
print(f"Jumlah baris: {form_df.shape[0]}")
print(f"Jumlah kolom: {form_df.shape[1]}")

display(form_df.head())

 Berhasil membaca Form Responses dari Google Spreadsheet sebagai teks.
Spreadsheet title : Recruitment Internship SOKO Financial Batch 15 (Responses)
Worksheet title   : Form Responses 1
Jumlah baris      : 271
Jumlah kolom      : 23
✅ form_df berhasil dibuat.
Jumlah baris: 271
Jumlah kolom: 23


,Timestamp,Email Address,Apakah kamu sudah membaca dan memahami Guidebook yang tersedia?,Email Aktif,Nama Lengkap,Jenis Kelamin,Domisili,Tanggal Lahir,Nomor WhatsApp Aktif,Pendidikan,...,Divisi Pertama,Posisi Divisi Pertama,Kenapa memilih divisi tersebut? Jelaskan alasannya!,Divisi Kedua,Posisi Divisi Kedua,Kenapa memilih divisi tersebut? Jelaskan alasannya!,CV Terbaru (PDF),Portfolio,Unggah Seluruh Bukti Persyaratan,Darimana Kamu Mendapat Informasi Mengenai OPREC Internship SOKO Financial Batch 15 ini?
0,7/13/2026 10:41:17,tommywijaya315@gmail.com,Sudah,Tommywijaya315@gmail.com,Tomi Wijaya,Laki-laki,"Kota Bekasi, Jawa Barat",9/24/2004,wa.me/6285388990284,Mahasiswa,...,HR Organizational Development,Staff,Karena sesuai dengan jurusan dan pengalaman saya,Business Development,Staff,Karena sesuai dengan jurusan dan pengalaman saya,https://drive.google.com/drive/folders/1NT1u91...,-,.,Instagram SOKO
1,7/13/2026 11:02:04,reginameidika30@gmail.com,Sudah,reginameidika30@gmail.com,Regina Meidika,Perempuan,"Kabupaten Tangerang, Banten",5/30/2003,081316537670,Fresh Graduate,...,HR Organizational Development,Staff,Saya memilih OD karena saya tertarik pada aspe...,HR Talent Acquisition,Staff,Saya memilih Talent Acquisition karena saya te...,https://drive.google.com/file/d/1vjWWEJFciU3yR...,-,https://drive.google.com/drive/folders/1ts-jkM...,Instagram SOKO
2,7/13/2026 11:05:54,anggaiansaputra06@gmail.com,Sudah,anggaiansaputra06@gmail.com,Angga Ian Saputra,Laki-laki,"Kota Bekasi, Jawa Barat",2/6/2003,wa.me/6287804032180,Fresh Graduate,...,HR Talent Acquisition,Staff,Saya memiliki dasar pengetahuan yang relevan k...,HR Talent Acquisition,Staff,Saya memiliki dasar pengetahuan yang relevan k...,https://drive.google.com/file/d/1yZkfmyV3MZDaL...,,https://drive.google.com/drive/folders/1UFlA3b...,Media Patner
3,7/13/2026 11:09:19,malestawitaka@gmail.com,Sudah,malestawitaka@gmail.com,Malesta Witaka Kussumasari,Perempuan,Yogyakarta,5/2/2006,wa.me/62895401664763,Mahasiswa,...,Data Analyst,Staff,Saya memilih divisi data analyst karena saya m...,Data Analyst,Staff,Saya memilih divisi data analyst karena saya m...,https://drive.google.com/drive/folders/1fIQTMF...,-,https://drive.google.com/drive/folders/1zKPSg3...,Instagram SOKO
4,7/13/2026 11:16:36,ineysafitri@gmail.com,Sudah,ineysafitri@gmail.com,Ine Safitri Akes Parema,Perempuan,"Kabupaten Pati, Jawa Tengah",2/22/2006,wa.me/6282314686358,Mahasiswa,...,HR Talent Acquisition,Staff,Saya memilih divisi HR Talent Acquisition kare...,HR Organizational Development,Staff,Saya memilih divisi Human Resource Organizatio...,https://drive.google.com/file/d/1HcZNURRqoIA7j...,-,https://drive.google.com/drive/folders/1bDP-FL...,Media Patner


In [ ]:

# CEK SHEET FILE INPUT PENDUKUNG


excel_files = {
    "Template Screening CV": TEMPLATE_SCREENING_FILE,
    "Skill Matrix": SKILL_MATRIX_FILE,
    "Academic Matrix": ACADEMIC_MATRIX_FILE,
    "Division Keywords": DIVISION_KEYWORDS_FILE,
}

print("=== DAFTAR SHEET SETIAP FILE INPUT PENDUKUNG ===")

for label, file_path in excel_files.items():
    if file_path.exists():
        sheet_names = get_excel_sheet_names(file_path)
        print(f"\n{label}:")
        for sheet in sheet_names:
            print(f"  - {sheet}")
    else:
        print(f"\n{label}:  file tidak ditemukan")

=== DAFTAR SHEET SETIAP FILE INPUT PENDUKUNG ===

Template Screening CV:
  - hasil_screening

Skill Matrix:
  - skill_matrix
  - summary
  - README

Academic Matrix:
  - academic_matrix
  - summary
  - README

Division Keywords:
  - division_keywords
  - summary
  - README


In [ ]:

# LOAD SKILL MATRIX


skill_df = load_excel_file(SKILL_MATRIX_FILE)

if skill_df is not None:
    print("\n=== PREVIEW SKILL MATRIX ===")
    display(skill_df.head())

 Berhasil membaca file: skill_matrix.xlsx
   Jumlah baris : 1021
   Jumlah kolom : 2

=== PREVIEW SKILL MATRIX ===


,division,skill_keyword
0,CEO Staff,strategic planning
1,CEO Staff,perencanaan strategis
2,CEO Staff,business strategy
3,CEO Staff,strategi bisnis
4,CEO Staff,organizational strategy


In [ ]:

# LOAD ACADEMIC MATRIX


academic_df = load_excel_file(ACADEMIC_MATRIX_FILE)

if academic_df is not None:
    print("\n=== PREVIEW ACADEMIC MATRIX ===")
    display(academic_df.head())

 Berhasil membaca file: academic_matrix.xlsx
   Jumlah baris : 466
   Jumlah kolom : 3

=== PREVIEW ACADEMIC MATRIX ===


,division,major_keyword,score
0,CEO Staff,Accounting and Management,5
1,CEO Staff,Administrasi Bisnis,5
2,CEO Staff,Bisnis,5
3,CEO Staff,Business,5
4,CEO Staff,Business Administration,5


In [ ]:

# LOAD DIVISION KEYWORDS


division_keywords_df = load_excel_file(DIVISION_KEYWORDS_FILE)

if division_keywords_df is not None:
    print("\n=== PREVIEW DIVISION KEYWORDS ===")
    display(division_keywords_df.head())

 Berhasil membaca file: division_keywords.xlsx
   Jumlah baris : 864
   Jumlah kolom : 2

=== PREVIEW DIVISION KEYWORDS ===


,division,keyword
0,CEO Staff,menyelaraskan arahan strategis
1,CEO Staff,align strategic direction
2,CEO Staff,strategic alignment
3,CEO Staff,CEO support
4,CEO Staff,supporting CEO


In [ ]:

# LOAD TEMPLATE SCREENING CV


template_sheets = get_excel_sheet_names(TEMPLATE_SCREENING_FILE)

print("=== TEMPLATE SCREENING CV ===")
print(f"File template: {TEMPLATE_SCREENING_FILE}")

if len(template_sheets) > 0:
    print("Sheet tersedia:")
    for sheet in template_sheets:
        print(f"- {sheet}")
else:
    print(" Tidak ada sheet yang terbaca dari template.")

if "hasil_screening" in template_sheets:
    print(" Sheet 'hasil_screening' ditemukan.")
else:
    print(" Sheet 'hasil_screening' belum ditemukan. Pastikan template sudah sesuai.")

=== TEMPLATE SCREENING CV ===
File template: /content/drive/Shareddrives/CV Screening Automation/01_input/template_screening_cv.xlsx
Sheet tersedia:
- hasil_screening
 Sheet 'hasil_screening' ditemukan.


In [ ]:

# VALIDASI STRUKTUR KOLOM MATRIX


def validate_required_columns(df, required_columns, file_label):
    """
    Mengecek apakah DataFrame memiliki semua kolom wajib.
    """
    if df is None:
        print(f" {file_label}: Data belum terbaca.")
        return False

    existing_columns = list(df.columns)
    missing_columns = [
        col for col in required_columns
        if col not in existing_columns
    ]

    if len(missing_columns) == 0:
        print(f" {file_label}: struktur kolom valid.")
        return True
    else:
        print(f" {file_label}: ada kolom yang belum tersedia.")
        print(f"   Kolom wajib     : {required_columns}")
        print(f"   Kolom tersedia  : {existing_columns}")
        print(f"   Kolom hilang    : {missing_columns}")
        return False


print("=== VALIDASI KOLOM MATRIX ===")

skill_valid = validate_required_columns(
    skill_df,
    ["division", "skill_keyword"],
    "Skill Matrix"
)

academic_valid = validate_required_columns(
    academic_df,
    ["division", "major_keyword", "score"],
    "Academic Matrix"
)

division_keywords_valid = validate_required_columns(
    division_keywords_df,
    ["division", "keyword"],
    "Division Keywords"
)

=== VALIDASI KOLOM MATRIX ===
 Skill Matrix: struktur kolom valid.
 Academic Matrix: struktur kolom valid.
 Division Keywords: struktur kolom valid.


In [ ]:
# CEK KOLOM FORM RESPONSES DENGAN AMAN

if "form_df" not in globals():
    print(" form_df belum tersedia.")
    print("Jalankan dulu cell load Form Responses dari Google Spreadsheet.")
elif form_df is None:
    print(" form_df masih None.")
    print("Cek apakah load Google Spreadsheet berhasil.")
else:
    print("=== DAFTAR KOLOM FORM RESPONSES ===")
    for i, col in enumerate(form_df.columns, start=1):
        print(f"{i}. {col}")

=== DAFTAR KOLOM FORM RESPONSES ===
1. Timestamp
2. Email Address
3. Apakah kamu sudah membaca dan memahami Guidebook yang tersedia?
4. Email Aktif
5. Nama Lengkap
6. Jenis Kelamin
7. Domisili
8. Tanggal Lahir
9. Nomor WhatsApp Aktif
10. Pendidikan
11. Jurusan dan Angkatan
12. Asal Instansi
13. Link Akun Instagram
14. Divisi Pertama 
15. Posisi Divisi Pertama
16. Kenapa memilih divisi tersebut? Jelaskan alasannya!
17. Divisi Kedua
18. Posisi Divisi Kedua
19. Kenapa memilih divisi tersebut? Jelaskan alasannya!
20. CV Terbaru (PDF)
21. Portfolio
22. Unggah Seluruh Bukti Persyaratan
23. Darimana Kamu Mendapat Informasi Mengenai OPREC Internship SOKO Financial Batch 15 ini?


In [ ]:
# RINGKASAN BAGIAN 2


print("=== RINGKASAN LOAD FILE ===")

load_status = {
    "Form Responses dari Google Spreadsheet": form_df is not None and not form_df.empty,
    "Skill Matrix": skill_df is not None and skill_valid,
    "Academic Matrix": academic_df is not None and academic_valid,
    "Division Keywords": division_keywords_df is not None and division_keywords_valid,
    "Template Screening CV": "hasil_screening" in template_sheets,
}

for label, status in load_status.items():
    if status:
        print(f" {label}: OK")
    else:
        print(f" {label}: Perlu dicek")

if all(load_status.values()):
    print("\n Semua file utama berhasil dibaca dan valid. Lanjut ke Bagian 3 — Cleaning Form Responses.")
else:
    print("\n Ada file atau struktur yang perlu dicek sebelum lanjut ke Bagian 3.")

=== RINGKASAN LOAD FILE ===
 Form Responses dari Google Spreadsheet: OK
 Skill Matrix: OK
 Academic Matrix: OK
 Division Keywords: OK
 Template Screening CV: OK

 Semua file utama berhasil dibaca dan valid. Lanjut ke Bagian 3 — Cleaning Form Responses.


In [ ]:
# BAGIAN 3 — CLEANING FORM RESPONSES

import re
import numpy as np

In [ ]:
# FUNGSI BANTU CLEANING

def normalize_column_name(col):
    """
    Membersihkan nama kolom:
    - ubah ke string
    - hapus spasi berlebih
    - rapikan newline
    """
    col = str(col)
    col = col.replace("\n", " ")
    col = re.sub(r"\s+", " ", col)
    return col.strip()

def clean_text(value):
    """
    Membersihkan isi cell teks.
    """
    if pd.isna(value):
        return ""

    value = str(value)
    value = value.replace("\n", " ")
    value = re.sub(r"\s+", " ", value)
    value = value.strip()

    if value.lower() in ["nan", "none", "null"]:
        return ""

    return value


def clean_optional_link(value):
    """
    Membersihkan link opsional.
    Jika isinya '-', kosong, atau nan, ubah menjadi string kosong.
    """
    value = clean_text(value)

    blank_values = [
        "",
        "-",
        "tidak ada",
        "none",
        "null",
        "nan",
        "n/a",
        "na"
    ]

    if value.lower() in blank_values:
        return ""

    return value


def extract_url_from_cell(value):
    """
    Mengambil URL dari cell.
    Bisa menangani:
    - URL biasa
    - HTML anchor dengan href
    """
    value = clean_text(value)

    if value == "":
        return ""

    # Jika berbentuk HTML anchor: https://...
    href_match = re.search(r'href=[^"\']+["\']', value)
    if href_match:
        return href_match.group(1).strip()

    # Jika sudah berupa URL biasa
    url_match = re.search(r'https?://[^\s<>"\']+', value)
    if url_match:
        return url_match.group(0).strip()

    return value.strip()


def is_blank(value):
    """
    Mengecek apakah value dianggap kosong.
    """
    value = clean_text(value)

    blank_values = [
        "",
        "-",
        "tidak ada",
        "none",
        "null",
        "nan",
        "n/a",
        "na"
    ]

    return value.lower() in blank_values

In [ ]:
# FUNGSI MEMBUAT NAMA KOLOM UNIK

def make_unique_columns(columns):
    """
    Membuat nama kolom menjadi unik.
    Jika ada duplikat:
    - Kolom pertama tetap nama asli
    - Kolom berikutnya menjadi nama_2, nama_3, dst.
    """
    seen = {}
    unique_columns = []

    for col in columns:
        col = normalize_column_name(col)

        if col not in seen:
            seen[col] = 1
            unique_columns.append(col)
        else:
            seen[col] += 1
            unique_columns.append(f"{col} {seen[col]}")

    return unique_columns

In [ ]:
# BERSIHKAN NAMA KOLOM FORM RESPONSES

if form_df is None:
    raise ValueError("form_df belum tersedia. Jalankan Bagian 2 terlebih dahulu.")

# Copy agar data asli tetap aman
raw_form_df = form_df.copy()

# Bersihkan nama kolom
raw_form_df.columns = make_unique_columns(raw_form_df.columns)

print("=== KOLOM FORM RESPONSES SETELAH DIBERSIHKAN ===")
for i, col in enumerate(raw_form_df.columns, start=1):
    print(f"{i}. {col}")

=== KOLOM FORM RESPONSES SETELAH DIBERSIHKAN ===
1. Timestamp
2. Email Address
3. Apakah kamu sudah membaca dan memahami Guidebook yang tersedia?
4. Email Aktif
5. Nama Lengkap
6. Jenis Kelamin
7. Domisili
8. Tanggal Lahir
9. Nomor WhatsApp Aktif
10. Pendidikan
11. Jurusan dan Angkatan
12. Asal Instansi
13. Link Akun Instagram
14. Divisi Pertama
15. Posisi Divisi Pertama
16. Kenapa memilih divisi tersebut? Jelaskan alasannya!
17. Divisi Kedua
18. Posisi Divisi Kedua
19. Kenapa memilih divisi tersebut? Jelaskan alasannya! 2
20. CV Terbaru (PDF)
21. Portfolio
22. Unggah Seluruh Bukti Persyaratan
23. Darimana Kamu Mendapat Informasi Mengenai OPREC Internship SOKO Financial Batch 15 ini?


In [ ]:
# FUNGSI DETEKSI KOLOM OTOMATIS

def find_best_column(df, candidate_names, label=None):
    """
    Mencari kolom berdasarkan urutan prioritas candidate_names.
    Jika kandidat pertama ditemukan dan punya isi, langsung dipakai.
    Aman untuk nama kolom yang sudah dibuat unik.
    """

    normalized_col_map = {
        normalize_column_name(col).lower(): col
        for col in df.columns
    }

    for candidate in candidate_names:
        candidate_norm = normalize_column_name(candidate).lower()

        if candidate_norm in normalized_col_map:
            col = normalized_col_map[candidate_norm]

            col_data = df[col]

            # Jika karena alasan tertentu col_data masih DataFrame,
            # ambil kolom pertama sebagai fallback.
            if isinstance(col_data, pd.DataFrame):
                col_data = col_data.iloc[:, 0]

            non_blank_count = col_data.apply(lambda x: not is_blank(x)).sum()

            if int(non_blank_count) > 0:
                return col

    return None

In [ ]:
# MAPPING KOLOM FORM RESPONSES

column_candidates = {
    "tanggal_submit": [
        "Timestamp",
        "Tanggal Submit",
        "w"
    ],
    "nama_lengkap": [
        "Nama Lengkap",
        "Nama"
    ],
    "nomor_whatsapp": [
        "Nomor WhatsApp Aktif",
        "Nomor WhatsApp",
        "WhatsApp",
        "No WhatsApp",
        "No. WhatsApp"
    ],
    "email": [
        "Email Aktif",
        "Email Address",
        "Email",
        "Alamat Email"
    ],
    "pendidikan": [
        "Pendidikan",
        "Status Pendidikan"
    ],
    "jurusan_angkatan": [
        "Jurusan dan Angkatan",
        "Jurusan",
        "Program Studi",
        "Jurusan / Angkatan"
    ],
    "asal_instansi": [
        "Asal Instansi",
        "Universitas",
        "Kampus",
        "Asal Kampus",
        "Nama Instansi"
    ],
    "instagram": [
        "Link Akun Instagram",
        "Instagram",
        "Akun Instagram",
        "Link Instagram"
    ],
    "pilihan_divisi_pertama": [
        "Divisi Pertama",
        "Pilihan Divisi Pertama",
        "Pilihan Divisi 1",
        "Division First Choice"
    ],
    "pilihan_posisi_pertama": [
        "Posisi Divisi Pertama",
        "Pilihan Posisi Pertama",
        "Posisi Pertama",
        "Pilihan Posisi 1"
    ],
    "pilihan_divisi_kedua": [
        "Divisi Kedua 2",
        "Divisi Kedua",
        "Pilihan Divisi Kedua",
        "Pilihan Divisi 2",
        "Division Second Choice"
    ],
    "pilihan_posisi_kedua": [
        "Posisi Divisi Kedua",
        "Pilihan Posisi Kedua",
        "Posisi Kedua",
        "Pilihan Posisi 2"
    ],
    "alasan_divisi_pertama": [
        "Kenapa memilih divisi tersebut? Jelaskan alasannya!",
        "Alasan Divisi Pertama",
        "Alasan Memilih Divisi Pertama",
        "Motivasi Divisi Pertama"
    ],
    "alasan_divisi_kedua": [
        "Kenapa memilih divisi tersebut? Jelaskan alasannya! 2",
        "Alasan Divisi Kedua",
        "Alasan Memilih Divisi Kedua",
        "Motivasi Divisi Kedua"
    ],
    "link_cv": [
        "CV Terbaru (PDF)",
        "Link CV",
        "CV",
        "Upload CV",
        "CV Terbaru"
    ],
    "link_portofolio": [
        "Portofolio",
        "Link Portofolio",
        "Portfolio",
        "Link Portfolio"
    ],
    "sumber_informasi": [
        "Darimana Kamu Mendapat Informasi Mengenai OPREC Internship SOKO Financial Batch 14 ini?",
        "Sumber Informasi",
        "Darimana kamu mendapat informasi",
        "Info Oprec"
    ]
}

detected_columns = {}

print("=== HASIL DETEKSI KOLOM ===")

for system_col, candidates in column_candidates.items():
    detected_col = find_best_column(raw_form_df, candidates, label=system_col)
    detected_columns[system_col] = detected_col

    if detected_col is not None:
        print(f"  {system_col}: {detected_col}")
    else:
        print(f"  {system_col}: tidak ditemukan")

=== HASIL DETEKSI KOLOM ===
  tanggal_submit: Timestamp
  nama_lengkap: Nama Lengkap
  nomor_whatsapp: Nomor WhatsApp Aktif
  email: Email Aktif
  pendidikan: Pendidikan
  jurusan_angkatan: Jurusan dan Angkatan
  asal_instansi: Asal Instansi
  instagram: Link Akun Instagram
  pilihan_divisi_pertama: Divisi Pertama
  pilihan_posisi_pertama: Posisi Divisi Pertama
  pilihan_divisi_kedua: Divisi Kedua
  pilihan_posisi_kedua: Posisi Divisi Kedua
  alasan_divisi_pertama: Kenapa memilih divisi tersebut? Jelaskan alasannya!
  alasan_divisi_kedua: Kenapa memilih divisi tersebut? Jelaskan alasannya! 2
  link_cv: CV Terbaru (PDF)
  link_portofolio: Portfolio
  sumber_informasi: tidak ditemukan


In [ ]:
# VALIDASI KOLOM WAJIB

required_system_columns = [
    "nama_lengkap",
    "nomor_whatsapp",
    "email",
    "pendidikan",
    "jurusan_angkatan",
    "asal_instansi",
    "pilihan_divisi_pertama",
    "pilihan_posisi_pertama",
    "pilihan_divisi_kedua",
    "pilihan_posisi_kedua",
    "link_cv"
]

missing_required_columns = [
    col for col in required_system_columns
    if detected_columns.get(col) is None
]

print("=== VALIDASI KOLOM WAJIB ===")

if len(missing_required_columns) == 0:
    print(" Semua kolom wajib berhasil terdeteksi.")
else:
    print(" Ada kolom wajib yang belum terdeteksi:")
    for col in missing_required_columns:
        print(f"- {col}")

    print("\nSilakan cek nama kolom Form Responses atau tambahkan alias kolom pada variable column_candidates.")

=== VALIDASI KOLOM WAJIB ===
 Semua kolom wajib berhasil terdeteksi.


In [ ]:
# BUAT DATAFRAME BERSIH INTERNAL

cleaned_df = pd.DataFrame()

def get_cleaned_column(df, detected_columns, system_col, default=""):
    """
    Mengambil kolom dari raw form berdasarkan hasil mapping.
    Jika kolom tidak ditemukan, isi default.
    """
    source_col = detected_columns.get(system_col)

    if source_col is None:
        return pd.Series([default] * len(df))

    return df[source_col].apply(clean_text)


cleaned_df["tanggal_submit"] = get_cleaned_column(raw_form_df, detected_columns, "tanggal_submit")
cleaned_df["nama_lengkap"] = get_cleaned_column(raw_form_df, detected_columns, "nama_lengkap")
cleaned_df["nomor_whatsapp"] = get_cleaned_column(raw_form_df, detected_columns, "nomor_whatsapp")
cleaned_df["email"] = get_cleaned_column(raw_form_df, detected_columns, "email")
cleaned_df["pendidikan"] = get_cleaned_column(raw_form_df, detected_columns, "pendidikan")
cleaned_df["jurusan_angkatan"] = get_cleaned_column(raw_form_df, detected_columns, "jurusan_angkatan")
cleaned_df["asal_instansi"] = get_cleaned_column(raw_form_df, detected_columns, "asal_instansi")
cleaned_df["instagram"] = get_cleaned_column(raw_form_df, detected_columns, "instagram")
cleaned_df["pilihan_divisi_pertama"] = get_cleaned_column(raw_form_df, detected_columns, "pilihan_divisi_pertama")
cleaned_df["pilihan_posisi_pertama"] = get_cleaned_column(raw_form_df, detected_columns, "pilihan_posisi_pertama")
cleaned_df["pilihan_divisi_kedua"] = get_cleaned_column(raw_form_df, detected_columns, "pilihan_divisi_kedua")
cleaned_df["pilihan_posisi_kedua"] = get_cleaned_column(raw_form_df, detected_columns, "pilihan_posisi_kedua")
cleaned_df["alasan_divisi_pertama"] = get_cleaned_column(
    raw_form_df,
    detected_columns,
    "alasan_divisi_pertama"
)

cleaned_df["alasan_divisi_kedua"] = get_cleaned_column(
    raw_form_df,
    detected_columns,
    "alasan_divisi_kedua"
)

cleaned_df["alasan_memilih_divisi"] = (
    cleaned_df["alasan_divisi_pertama"].fillna("").astype(str)
    + " "
    + cleaned_df["alasan_divisi_kedua"].fillna("").astype(str)
).apply(clean_text)

cleaned_df["link_cv"] = (get_cleaned_column(raw_form_df, detected_columns, "link_cv").apply(clean_optional_link).apply(extract_url_from_cell))
cleaned_df["link_portofolio"] = (get_cleaned_column(raw_form_df, detected_columns, "link_portofolio").apply(clean_optional_link).apply(extract_url_from_cell))

cleaned_df["sumber_informasi"] = get_cleaned_column(raw_form_df, detected_columns, "sumber_informasi")

# Tambahkan nomor urut
cleaned_df.insert(0, "no", range(1, len(cleaned_df) + 1))

print(" cleaned_df berhasil dibuat.")
print(f"Jumlah kandidat: {len(cleaned_df)}")

display(cleaned_df.head())

 cleaned_df berhasil dibuat.
Jumlah kandidat: 271


,no,tanggal_submit,nama_lengkap,nomor_whatsapp,email,pendidikan,jurusan_angkatan,asal_instansi,instagram,pilihan_divisi_pertama,pilihan_posisi_pertama,pilihan_divisi_kedua,pilihan_posisi_kedua,alasan_divisi_pertama,alasan_divisi_kedua,alasan_memilih_divisi,link_cv,link_portofolio,sumber_informasi
0,1,7/13/2026 10:41:17,Tomi Wijaya,wa.me/6285388990284,Tommywijaya315@gmail.com,Mahasiswa,Manajemen - 2022,Universitas Bhayangkara Jakarta Raya,https://www.instagram.com/tommywijy/,HR Organizational Development,Staff,Business Development,Staff,Karena sesuai dengan jurusan dan pengalaman saya,Karena sesuai dengan jurusan dan pengalaman saya,Karena sesuai dengan jurusan dan pengalaman sa...,https://drive.google.com/drive/folders/1NT1u91...,,
1,2,7/13/2026 11:02:04,Regina Meidika,081316537670,reginameidika30@gmail.com,Fresh Graduate,Hukum - 2026,Universitas Lampung,https://www.instagram.com/reginameii_?igsh=MWc...,HR Organizational Development,Staff,HR Talent Acquisition,Staff,Saya memilih OD karena saya tertarik pada aspe...,Saya memilih Talent Acquisition karena saya te...,Saya memilih OD karena saya tertarik pada aspe...,https://drive.google.com/file/d/1vjWWEJFciU3yR...,,
2,3,7/13/2026 11:05:54,Angga Ian Saputra,wa.me/6287804032180,anggaiansaputra06@gmail.com,Fresh Graduate,Manajemen - 2021,Universitas Bhayangkara Jakarta Raya,instagram.com/sanggaian,HR Talent Acquisition,Staff,HR Talent Acquisition,Staff,Saya memiliki dasar pengetahuan yang relevan k...,Saya memiliki dasar pengetahuan yang relevan k...,Saya memiliki dasar pengetahuan yang relevan k...,https://drive.google.com/file/d/1yZkfmyV3MZDaL...,,
3,4,7/13/2026 11:09:19,Malesta Witaka Kussumasari,wa.me/62895401664763,malestawitaka@gmail.com,Mahasiswa,Akuntansi - 2024,UPN Veteran Yogyakarta,instagram.com/maalesta,Data Analyst,Staff,Data Analyst,Staff,Saya memilih divisi data analyst karena saya m...,Saya memilih divisi data analyst karena saya m...,Saya memilih divisi data analyst karena saya m...,https://drive.google.com/drive/folders/1fIQTMF...,,
4,5,7/13/2026 11:16:36,Ine Safitri Akes Parema,wa.me/6282314686358,ineysafitri@gmail.com,Mahasiswa,Psikologi - 2023,Universitas Negeri Semarang,instagram.com/h0neyy_neyy,HR Talent Acquisition,Staff,HR Organizational Development,Staff,Saya memilih divisi HR Talent Acquisition kare...,Saya memilih divisi Human Resource Organizatio...,Saya memilih divisi HR Talent Acquisition kare...,https://drive.google.com/file/d/1HcZNURRqoIA7j...,,


In [ ]:
# BERSIHKAN NAMA DIVISI DAN POSISI

division_columns = [
    "pilihan_divisi_pertama",
    "pilihan_divisi_kedua"
]

position_columns = [
    "pilihan_posisi_pertama",
    "pilihan_posisi_kedua"
]

for col in division_columns + position_columns:
    cleaned_df[col] = cleaned_df[col].apply(clean_text)

print("=== SAMPLE PILIHAN DIVISI ===")
display(
    cleaned_df[
        [
            "nama_lengkap",
            "pilihan_divisi_pertama",
            "pilihan_posisi_pertama",
            "pilihan_divisi_kedua",
            "pilihan_posisi_kedua"
        ]
    ].head()
)

=== SAMPLE PILIHAN DIVISI ===


,nama_lengkap,pilihan_divisi_pertama,pilihan_posisi_pertama,pilihan_divisi_kedua,pilihan_posisi_kedua
0,Tomi Wijaya,HR Organizational Development,Staff,Business Development,Staff
1,Regina Meidika,HR Organizational Development,Staff,HR Talent Acquisition,Staff
2,Angga Ian Saputra,HR Talent Acquisition,Staff,HR Talent Acquisition,Staff
3,Malesta Witaka Kussumasari,Data Analyst,Staff,Data Analyst,Staff
4,Ine Safitri Akes Parema,HR Talent Acquisition,Staff,HR Organizational Development,Staff


In [ ]:
# CEK DAFTAR DIVISI DARI FORM RESPONSES

divisi_pertama_list = sorted(
    cleaned_df["pilihan_divisi_pertama"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace("", np.nan)
    .dropna()
    .unique()
)

divisi_kedua_list = sorted(
    cleaned_df["pilihan_divisi_kedua"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace("", np.nan)
    .dropna()
    .unique()
)

all_form_divisions = sorted(set(divisi_pertama_list + divisi_kedua_list))

print("=== DAFTAR DIVISI DARI FORM RESPONSES ===")
for div in all_form_divisions:
    print(f"- {div}")

=== DAFTAR DIVISI DARI FORM RESPONSES ===
- Business Development
- CEO Staff
- Content Creator
- Customer Services
- Data Analyst
- Digital Marketing
- Event Project
- Graphic Design
- HR Organizational Development
- HR Talent Acquisition
- Social Media Specialist


In [ ]:
# CEK KESESUAIAN DIVISI FORM DENGAN MATRIX

def get_matrix_divisions(df, division_col="division"):
    if df is None or division_col not in df.columns:
        return []

    return sorted(
        df[division_col]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
        .unique()
    )


skill_matrix_divisions = get_matrix_divisions(skill_df)
academic_matrix_divisions = get_matrix_divisions(academic_df)
division_keyword_divisions = get_matrix_divisions(division_keywords_df)

matrix_division_sets = {
    "Skill Matrix": set(skill_matrix_divisions),
    "Academic Matrix": set(academic_matrix_divisions),
    "Division Keywords": set(division_keyword_divisions),
}

print("=== CEK DIVISI FORM VS MATRIX ===")

for matrix_name, matrix_divisions in matrix_division_sets.items():
    missing_in_matrix = sorted(set(all_form_divisions) - matrix_divisions)

    print(f"\n{matrix_name}:")
    if len(missing_in_matrix) == 0:
        print(" Semua divisi dari Form Responses ditemukan di matrix.")
    else:
        print(" Ada divisi dari Form Responses yang belum ada di matrix:")
        for div in missing_in_matrix:
            print(f"- {div}")

=== CEK DIVISI FORM VS MATRIX ===

Skill Matrix:
 Semua divisi dari Form Responses ditemukan di matrix.

Academic Matrix:
 Semua divisi dari Form Responses ditemukan di matrix.

Division Keywords:
 Semua divisi dari Form Responses ditemukan di matrix.


In [ ]:
# BUAT OUTPUT DATAFRAME SESUAI TEMPLATE HASIL_SCREENING

output_columns = [
    "No",
    "Nama Lengkap",
    "Nomor WhatsApp",
    "Email",
    "Pendidikan",
    "Jurusan dan Angkatan",
    "Asal Instansi",
    "Instagram",
    "Pilihan Divisi Pertama",
    "Pilihan Posisi Pertama",
    "Pilihan Divisi Kedua",
    "Pilihan Posisi Kedua",
    "Link CV",
    "Link Portofolio",
    "Status Administrasi",
    "Flag",
    "Skor Divisi Pertama",
    "Skor Divisi Kedua",
    "Rekomendasi Divisi",
    "Rekomendasi Status",
    "Alasan Otomatis",
    "Catatan Reviewer",
    "Status Final Reviewer",
    "Alasan Final Reviewer"
]

screening_output_df = pd.DataFrame()

screening_output_df["No"] = cleaned_df["no"]
screening_output_df["Nama Lengkap"] = cleaned_df["nama_lengkap"]
screening_output_df["Nomor WhatsApp"] = cleaned_df["nomor_whatsapp"]
screening_output_df["Email"] = cleaned_df["email"]
screening_output_df["Pendidikan"] = cleaned_df["pendidikan"]
screening_output_df["Jurusan dan Angkatan"] = cleaned_df["jurusan_angkatan"]
screening_output_df["Asal Instansi"] = cleaned_df["asal_instansi"]
screening_output_df["Instagram"] = cleaned_df["instagram"]
screening_output_df["Pilihan Divisi Pertama"] = cleaned_df["pilihan_divisi_pertama"]
screening_output_df["Pilihan Posisi Pertama"] = cleaned_df["pilihan_posisi_pertama"]
screening_output_df["Pilihan Divisi Kedua"] = cleaned_df["pilihan_divisi_kedua"]
screening_output_df["Pilihan Posisi Kedua"] = cleaned_df["pilihan_posisi_kedua"]
screening_output_df["Link CV"] = cleaned_df["link_cv"]
screening_output_df["Link Portofolio"] = cleaned_df["link_portofolio"]

# Kolom berikut akan diisi di bagian selanjutnya
screening_output_df["Status Administrasi"] = ""
screening_output_df["Flag"] = ""
screening_output_df["Skor Divisi Pertama"] = ""
screening_output_df["Skor Divisi Kedua"] = ""
screening_output_df["Rekomendasi Divisi"] = ""
screening_output_df["Rekomendasi Status"] = ""
screening_output_df["Alasan Otomatis"] = ""
screening_output_df["Catatan Reviewer"] = ""
screening_output_df["Status Final Reviewer"] = ""
screening_output_df["Alasan Final Reviewer"] = ""

# Pastikan urutan kolom sesuai template
screening_output_df = screening_output_df[output_columns]

print(" screening_output_df berhasil dibuat.")
display(screening_output_df.head())

 screening_output_df berhasil dibuat.


,No,Nama Lengkap,Nomor WhatsApp,Email,Pendidikan,Jurusan dan Angkatan,Asal Instansi,Instagram,Pilihan Divisi Pertama,Pilihan Posisi Pertama,...,Status Administrasi,Flag,Skor Divisi Pertama,Skor Divisi Kedua,Rekomendasi Divisi,Rekomendasi Status,Alasan Otomatis,Catatan Reviewer,Status Final Reviewer,Alasan Final Reviewer
0,1,Tomi Wijaya,wa.me/6285388990284,Tommywijaya315@gmail.com,Mahasiswa,Manajemen - 2022,Universitas Bhayangkara Jakarta Raya,https://www.instagram.com/tommywijy/,HR Organizational Development,Staff,...,,,,,,,,,,
1,2,Regina Meidika,081316537670,reginameidika30@gmail.com,Fresh Graduate,Hukum - 2026,Universitas Lampung,https://www.instagram.com/reginameii_?igsh=MWc...,HR Organizational Development,Staff,...,,,,,,,,,,
2,3,Angga Ian Saputra,wa.me/6287804032180,anggaiansaputra06@gmail.com,Fresh Graduate,Manajemen - 2021,Universitas Bhayangkara Jakarta Raya,instagram.com/sanggaian,HR Talent Acquisition,Staff,...,,,,,,,,,,
3,4,Malesta Witaka Kussumasari,wa.me/62895401664763,malestawitaka@gmail.com,Mahasiswa,Akuntansi - 2024,UPN Veteran Yogyakarta,instagram.com/maalesta,Data Analyst,Staff,...,,,,,,,,,,
4,5,Ine Safitri Akes Parema,wa.me/6282314686358,ineysafitri@gmail.com,Mahasiswa,Psikologi - 2023,Universitas Negeri Semarang,instagram.com/h0neyy_neyy,HR Talent Acquisition,Staff,...,,,,,,,,,,


In [ ]:
# RINGKASAN BAGIAN 3

print("=== RINGKASAN CLEANING FORM RESPONSES ===")
print(f"Jumlah kandidat terbaca       : {len(cleaned_df)}")
print(f"Jumlah kolom internal         : {cleaned_df.shape[1]}")
print(f"Jumlah kolom output utama     : {screening_output_df.shape[1]}")

print("\nKolom internal yang tersedia:")
for col in cleaned_df.columns:
    print(f"- {col}")

print("\n Bagian 3 selesai.")
print("Lanjut ke Bagian 4 — Validasi Administrasi dan Flag.")

=== RINGKASAN CLEANING FORM RESPONSES ===
Jumlah kandidat terbaca       : 271
Jumlah kolom internal         : 19
Jumlah kolom output utama     : 24

Kolom internal yang tersedia:
- no
- tanggal_submit
- nama_lengkap
- nomor_whatsapp
- email
- pendidikan
- jurusan_angkatan
- asal_instansi
- instagram
- pilihan_divisi_pertama
- pilihan_posisi_pertama
- pilihan_divisi_kedua
- pilihan_posisi_kedua
- alasan_divisi_pertama
- alasan_divisi_kedua
- alasan_memilih_divisi
- link_cv
- link_portofolio
- sumber_informasi

 Bagian 3 selesai.
Lanjut ke Bagian 4 — Validasi Administrasi dan Flag.


In [ ]:
# BAGIAN 4 — VALIDASI ADMINISTRASI DAN FLAG

In [ ]:
# KONFIGURASI VALIDASI DAN FLAG

# Divisi yang wajib memiliki portofolio
CREATIVE_DIVISIONS = [
    "Graphic Design",
    "Content Creator"
]

# Aktifkan pengecekan bukti administrasi jika form oprec mewajibkan bukti follow/repost/tag
ADMIN_PROOF_REQUIRED = True

# Keyword untuk mendeteksi kolom bukti administrasi pada Form Responses
ADMIN_PROOF_COLUMN_KEYWORDS = [
    "bukti",
    "follow",
    "subscribe",
    "repost",
    "tag"
]

# Kolom bukti yang TIDAK ingin dianggap bukti administrasi
# Misalnya jika ada kolom CV/Portofolio yang mengandung kata upload.
EXCLUDE_ADMIN_PROOF_KEYWORDS = [
    "cv",
    "portofolio",
    "portfolio"
]

In [ ]:
# FUNGSI BANTU VALIDASI

def is_url(value):
    """
    Mengecek apakah value terlihat seperti URL.
    """
    value = clean_text(value).lower()
    return value.startswith("http://") or value.startswith("https://")


def is_google_drive_folder(value):
    """
    Mengecek apakah link adalah Google Drive folder.
    """
    value = clean_text(value).lower()
    return "drive.google.com" in value and "/folders/" in value


def is_google_drive_file(value):
    """
    Mengecek apakah link adalah Google Drive file.
    """
    value = clean_text(value).lower()
    return "drive.google.com" in value and ("/file/d/" in value or "open?id=" in value)


def same_text(a, b):
    """
    Mengecek apakah dua teks sama setelah dibersihkan dan dibuat lowercase.
    """
    return clean_text(a).lower() == clean_text(b).lower()


def add_flag(flags, flag_name):
    """
    Menambahkan flag jika belum ada.
    """
    if flag_name not in flags:
        flags.append(flag_name)
    return flags

In [ ]:
# DETEKSI KOLOM BUKTI ADMINISTRASI

admin_proof_columns = []

for col in raw_form_df.columns:
    col_lower = clean_text(col).lower()

    contains_required_keyword = any(keyword in col_lower for keyword in ADMIN_PROOF_COLUMN_KEYWORDS)
    contains_excluded_keyword = any(keyword in col_lower for keyword in EXCLUDE_ADMIN_PROOF_KEYWORDS)

    if contains_required_keyword and not contains_excluded_keyword:
        admin_proof_columns.append(col)

print("=== KOLOM BUKTI ADMINISTRASI TERDETEKSI ===")

if len(admin_proof_columns) > 0:
    for col in admin_proof_columns:
        print(f"- {col}")
else:
    print(" Tidak ada kolom bukti administrasi yang terdeteksi.")
    print("Jika form oprec memang tidak memakai bukti administrasi, set ADMIN_PROOF_REQUIRED = False.")

=== KOLOM BUKTI ADMINISTRASI TERDETEKSI ===
- Link Akun Instagram
- Unggah Seluruh Bukti Persyaratan


In [ ]:
# FUNGSI VALIDASI PER KANDIDAT

def validate_candidate(row, raw_row=None):
    """
    Melakukan validasi administrasi untuk satu kandidat.
    Mengembalikan:
    - status_administrasi
    - flag string
    """

    flags = []

    # ----------------------------
    # 1. Cek data wajib
    # ----------------------------
    required_fields = {
        "Nama Lengkap": row.get("nama_lengkap", ""),
        "Nomor WhatsApp": row.get("nomor_whatsapp", ""),
        "Email": row.get("email", ""),
        "Pendidikan": row.get("pendidikan", ""),
        "Jurusan dan Angkatan": row.get("jurusan_angkatan", ""),
        "Asal Instansi": row.get("asal_instansi", ""),
        "Pilihan Divisi Pertama": row.get("pilihan_divisi_pertama", ""),
        "Pilihan Posisi Pertama": row.get("pilihan_posisi_pertama", ""),
        "Pilihan Divisi Kedua": row.get("pilihan_divisi_kedua", ""),
        "Pilihan Posisi Kedua": row.get("pilihan_posisi_kedua", ""),
        "Link CV": row.get("link_cv", "")
    }

    missing_required = []

    for field_name, field_value in required_fields.items():
        if is_blank(field_value):
            missing_required.append(field_name)

    if len(missing_required) > 0:
        add_flag(flags, "MISSING_REQUIRED_DATA")

    # ----------------------------
    # 2. Cek CV
    # ----------------------------
    link_cv = row.get("link_cv", "")

    if is_blank(link_cv):
        add_flag(flags, "MISSING_CV")
    else:
        if not is_url(link_cv):
            add_flag(flags, "INVALID_CV_LINK")

        if is_google_drive_folder(link_cv):
            add_flag(flags, "CV_LINK_IS_FOLDER")

    # ----------------------------
    # 3. Cek portofolio untuk divisi kreatif
    # ----------------------------
    divisi_pertama = clean_text(row.get("pilihan_divisi_pertama", ""))
    divisi_kedua = clean_text(row.get("pilihan_divisi_kedua", ""))
    link_portofolio = row.get("link_portofolio", "")

    memilih_divisi_kreatif = (
        divisi_pertama in CREATIVE_DIVISIONS or
        divisi_kedua in CREATIVE_DIVISIONS
    )

    if memilih_divisi_kreatif and is_blank(link_portofolio):
        add_flag(flags, "MISSING_PORTFOLIO_FOR_CREATIVE")

    # ----------------------------
    # 4. Cek bukti administrasi
    # ----------------------------
    if ADMIN_PROOF_REQUIRED and raw_row is not None and len(admin_proof_columns) > 0:
        missing_admin_proofs = []

        for proof_col in admin_proof_columns:
            proof_value = raw_row.get(proof_col, "")
            if is_blank(proof_value):
                missing_admin_proofs.append(proof_col)

        if len(missing_admin_proofs) > 0:
            add_flag(flags, "MISSING_ADMIN_PROOF")

    # ----------------------------
    # 5. Tentukan status administrasi
    # ----------------------------
    critical_flags = [
        "MISSING_REQUIRED_DATA",
        "MISSING_CV",
        "INVALID_CV_LINK",
        "MISSING_ADMIN_PROOF"
    ]

    check_flags = [
        "CV_LINK_IS_FOLDER",
        "MISSING_PORTFOLIO_FOR_CREATIVE"
    ]

    if any(flag in flags for flag in critical_flags):
        status_administrasi = "Tidak Lengkap"
    elif any(flag in flags for flag in check_flags):
        status_administrasi = "Perlu Dicek"
    else:
        status_administrasi = "Administrasi Lengkap"

    flag_text = "; ".join(flags)

    return status_administrasi, flag_text

In [ ]:
# JALANKAN VALIDASI ADMINISTRASI

status_list = []
flag_list = []

for idx, row in cleaned_df.iterrows():
    raw_row = raw_form_df.iloc[idx] if idx < len(raw_form_df) else None

    status_administrasi, flag_text = validate_candidate(
        row=row,
        raw_row=raw_row
    )

    status_list.append(status_administrasi)
    flag_list.append(flag_text)

cleaned_df["status_administrasi"] = status_list
cleaned_df["flag"] = flag_list

print(" Validasi administrasi selesai.")
display(
    cleaned_df[
        [
            "no",
            "nama_lengkap",
            "pilihan_divisi_pertama",
            "pilihan_divisi_kedua",
            "link_cv",
            "link_portofolio",
            "status_administrasi",
            "flag"
        ]
    ].head()
)

 Validasi administrasi selesai.


,no,nama_lengkap,pilihan_divisi_pertama,pilihan_divisi_kedua,link_cv,link_portofolio,status_administrasi,flag
0,1,Tomi Wijaya,HR Organizational Development,Business Development,https://drive.google.com/drive/folders/1NT1u91...,,Perlu Dicek,CV_LINK_IS_FOLDER
1,2,Regina Meidika,HR Organizational Development,HR Talent Acquisition,https://drive.google.com/file/d/1vjWWEJFciU3yR...,,Administrasi Lengkap,
2,3,Angga Ian Saputra,HR Talent Acquisition,HR Talent Acquisition,https://drive.google.com/file/d/1yZkfmyV3MZDaL...,,Administrasi Lengkap,
3,4,Malesta Witaka Kussumasari,Data Analyst,Data Analyst,https://drive.google.com/drive/folders/1fIQTMF...,,Perlu Dicek,CV_LINK_IS_FOLDER
4,5,Ine Safitri Akes Parema,HR Talent Acquisition,HR Organizational Development,https://drive.google.com/file/d/1HcZNURRqoIA7j...,,Administrasi Lengkap,


In [ ]:
# UPDATE OUTPUT DATAFRAME DENGAN STATUS DAN FLAG

screening_output_df["Status Administrasi"] = cleaned_df["status_administrasi"]
screening_output_df["Flag"] = cleaned_df["flag"]

print(" screening_output_df berhasil diperbarui dengan Status Administrasi dan Flag.")
display(screening_output_df.head())

 screening_output_df berhasil diperbarui dengan Status Administrasi dan Flag.


,No,Nama Lengkap,Nomor WhatsApp,Email,Pendidikan,Jurusan dan Angkatan,Asal Instansi,Instagram,Pilihan Divisi Pertama,Pilihan Posisi Pertama,...,Status Administrasi,Flag,Skor Divisi Pertama,Skor Divisi Kedua,Rekomendasi Divisi,Rekomendasi Status,Alasan Otomatis,Catatan Reviewer,Status Final Reviewer,Alasan Final Reviewer
0,1,Tomi Wijaya,wa.me/6285388990284,Tommywijaya315@gmail.com,Mahasiswa,Manajemen - 2022,Universitas Bhayangkara Jakarta Raya,https://www.instagram.com/tommywijy/,HR Organizational Development,Staff,...,Perlu Dicek,CV_LINK_IS_FOLDER,,,,,,,,
1,2,Regina Meidika,081316537670,reginameidika30@gmail.com,Fresh Graduate,Hukum - 2026,Universitas Lampung,https://www.instagram.com/reginameii_?igsh=MWc...,HR Organizational Development,Staff,...,Administrasi Lengkap,,,,,,,,,
2,3,Angga Ian Saputra,wa.me/6287804032180,anggaiansaputra06@gmail.com,Fresh Graduate,Manajemen - 2021,Universitas Bhayangkara Jakarta Raya,instagram.com/sanggaian,HR Talent Acquisition,Staff,...,Administrasi Lengkap,,,,,,,,,
3,4,Malesta Witaka Kussumasari,wa.me/62895401664763,malestawitaka@gmail.com,Mahasiswa,Akuntansi - 2024,UPN Veteran Yogyakarta,instagram.com/maalesta,Data Analyst,Staff,...,Perlu Dicek,CV_LINK_IS_FOLDER,,,,,,,,
4,5,Ine Safitri Akes Parema,wa.me/6282314686358,ineysafitri@gmail.com,Mahasiswa,Psikologi - 2023,Universitas Negeri Semarang,instagram.com/h0neyy_neyy,HR Talent Acquisition,Staff,...,Administrasi Lengkap,,,,,,,,,


In [ ]:
# RINGKASAN STATUS ADMINISTRASI

print("=== RINGKASAN STATUS ADMINISTRASI ===")

status_summary = cleaned_df["status_administrasi"].value_counts(dropna=False)

display(status_summary)

=== RINGKASAN STATUS ADMINISTRASI ===


,count
status_administrasi,
Administrasi Lengkap,207
Perlu Dicek,63
Tidak Lengkap,1


In [ ]:
# RINGKASAN FLAG

print("=== RINGKASAN FLAG ===")

all_flags = []

for flag_text in cleaned_df["flag"]:
    if not is_blank(flag_text):
        split_flags = [flag.strip() for flag in flag_text.split(";") if flag.strip()]
        all_flags.extend(split_flags)

if len(all_flags) > 0:
    flag_summary = pd.Series(all_flags).value_counts()
    display(flag_summary)
else:
    print(" Tidak ada flag yang muncul.")

=== RINGKASAN FLAG ===


,count
CV_LINK_IS_FOLDER,63
MISSING_ADMIN_PROOF,1


In [ ]:
# DATAFRAME KANDIDAT DENGAN FLAG

flagged_candidates_df = cleaned_df[
    cleaned_df["flag"].apply(lambda x: not is_blank(x))
].copy()

if len(flagged_candidates_df) > 0:
    flagged_candidates_view = flagged_candidates_df[
        [
            "no",
            "nama_lengkap",
            "asal_instansi",
            "pilihan_divisi_pertama",
            "pilihan_divisi_kedua",
            "status_administrasi",
            "flag"
        ]
    ]

    print(f" Jumlah kandidat dengan flag: {len(flagged_candidates_df)}")
    display(flagged_candidates_view.head(20))
else:
    print(" Tidak ada kandidat dengan flag.")

 Jumlah kandidat dengan flag: 64


,no,nama_lengkap,asal_instansi,pilihan_divisi_pertama,pilihan_divisi_kedua,status_administrasi,flag
0,1,Tomi Wijaya,Universitas Bhayangkara Jakarta Raya,HR Organizational Development,Business Development,Perlu Dicek,CV_LINK_IS_FOLDER
3,4,Malesta Witaka Kussumasari,UPN Veteran Yogyakarta,Data Analyst,Data Analyst,Perlu Dicek,CV_LINK_IS_FOLDER
5,6,Muhammad Rafli Choirul Umam,Universitas Negeri Jakarta,Data Analyst,Data Analyst,Perlu Dicek,CV_LINK_IS_FOLDER
8,9,Suhairi Ibnu Abdillah,Universitas Bojonegoro,Graphic Design,Content Creator,Perlu Dicek,CV_LINK_IS_FOLDER
13,14,Zacki Ferdinansyah,Universitas Esa Unggul,Data Analyst,Data Analyst,Perlu Dicek,CV_LINK_IS_FOLDER
14,15,Dwi Musalim Mudrik,Sekolah Tinggi Ilmu Administrasi AAN Yogyakarta,Business Development,Customer Services,Perlu Dicek,CV_LINK_IS_FOLDER
16,17,DITA DWI RIYANI,UNIVERSITAS AIRLANGGA,Business Development,Event Project,Perlu Dicek,CV_LINK_IS_FOLDER
29,30,Jihan Nabilah Rahman,Universitas Bakrie,Data Analyst,Business Development,Perlu Dicek,CV_LINK_IS_FOLDER
32,33,Fadhil Muhammad,Bina Insani University,Data Analyst,Social Media Specialist,Perlu Dicek,CV_LINK_IS_FOLDER
34,35,Ester Tambunan,"UPN ""Veteran"" Yogyakarta",Event Project,Business Development,Perlu Dicek,CV_LINK_IS_FOLDER


In [ ]:
# RINGKASAN BAGIAN 4

print("=== RINGKASAN BAGIAN 4 ===")
print(f"Jumlah kandidat total        : {len(cleaned_df)}")
print(f"Jumlah kandidat dengan flag  : {len(flagged_candidates_df)}")

print("\nStatus Administrasi:")
display(cleaned_df["status_administrasi"].value_counts(dropna=False))

print("\n Bagian 4 selesai.")
print("Lanjut ke Bagian 5 — Matching Matrix dan Scoring Awal.")

=== RINGKASAN BAGIAN 4 ===
Jumlah kandidat total        : 271
Jumlah kandidat dengan flag  : 64

Status Administrasi:


,count
status_administrasi,
Administrasi Lengkap,207
Perlu Dicek,63
Tidak Lengkap,1



 Bagian 4 selesai.
Lanjut ke Bagian 5 — Matching Matrix dan Scoring Awal.


In [ ]:

# BAGIAN 5 — DOWNLOAD DAN EKSTRAKSI ISI CV


!pip install -q pymupdf gdown

import re
import fitz  # PyMuPDF
import gdown
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 42.4 MB/s eta 0:00:00


In [ ]:

# KONFIGURASI EKSTRAKSI CV


# Minimal panjang teks agar CV dianggap terbaca.
# Jika di bawah angka ini, kemungkinan CV adalah scan/gambar atau ekstraksi gagal sebagian.
MIN_CV_TEXT_LENGTH = 300

# Pastikan folder download dan folder text sudah tersedia.
CV_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
CV_TEXT_DIR.mkdir(parents=True, exist_ok=True)

print("=== KONFIGURASI EKSTRAKSI CV ===")
print(f"CV_DOWNLOAD_DIR       : {CV_DOWNLOAD_DIR}")
print(f"CV_TEXT_DIR           : {CV_TEXT_DIR}")
print(f"MIN_CV_TEXT_LENGTH    : {MIN_CV_TEXT_LENGTH}")


=== KONFIGURASI EKSTRAKSI CV ===
CV_DOWNLOAD_DIR       : /content/drive/Shareddrives/CV Screening Automation/04_temp_cv_downloads
CV_TEXT_DIR           : /content/drive/Shareddrives/CV Screening Automation/05_cv_text
MIN_CV_TEXT_LENGTH    : 300


In [ ]:

# FUNGSI CLEANING LINK CV


def extract_url_from_cell(value):
    """
    Mengambil URL dari cell.
    Bisa menangani:
    - URL biasa
    - HTML anchor seperti ......</a>
    """
    value = clean_text(value)

    if value == "":
        return ""

    # Jika berbentuk HTML anchor, ambil isi href
    href_match = re.search(r'href=[^"\\\']+["\\\']', value)
    if href_match:
        return href_match.group(1).strip()

    # Jika berbentuk URL biasa
    url_match = re.search(r'https?://[^\s<>"\\\']+', value)
    if url_match:
        return url_match.group(0).strip()

    return value.strip()


def extract_google_drive_file_id(url):
    """
    Mengambil Google Drive file ID dari URL.
    Mendukung format:
    - drive.google.com/file/d/FILE_ID/view
    - drive.google.com/open?id=FILE_ID
    - drive.google.com/uc?id=FILE_ID
    """
    url = clean_text(url)

    patterns = [
        r"/file/d/([^/]+)",
        r"[?&]id=([^&]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1).strip()

    return ""

In [ ]:

# FUNGSI DOWNLOAD CV


def safe_filename(text):
    """
    Membuat nama file aman dari nama kandidat.
    """
    text = clean_text(text)
    text = re.sub(r"[^a-zA-Z0-9_\- ]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text[:80] if text else "unknown_candidate"


def download_cv_from_drive(file_id, output_path):
    """
    Download file Google Drive menggunakan file_id.
    """
    try:
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, str(output_path), quiet=True)

        if output_path.exists() and output_path.stat().st_size > 0:
            return True, "Download success"
        else:
            return False, "Downloaded file is empty or not found"

    except Exception as e:
        return False, str(e)

In [ ]:

# FUNGSI EKSTRAKSI TEKS PDF


def extract_text_from_pdf(pdf_path):
    """
    Ekstrak teks dari file PDF menggunakan PyMuPDF.
    """
    try:
        text = ""

        with fitz.open(pdf_path) as doc:
            for page in doc:
                text += page.get_text("text") + "\n"

        text = re.sub(r"\s+", " ", text).strip()
        return text, "Extraction success"

    except Exception as e:
        return "", str(e)

In [ ]:

# FUNGSI UPDATE FLAG


def append_flag_to_text(existing_flag_text, new_flag):
    """
    Menambahkan flag baru ke string flag yang dipisahkan semicolon.
    """
    existing_flag_text = clean_text(existing_flag_text)

    if existing_flag_text == "":
        return new_flag

    flags = [flag.strip() for flag in existing_flag_text.split(";") if flag.strip()]

    if new_flag not in flags:
        flags.append(new_flag)

    return "; ".join(flags)

In [ ]:

# PROSES DOWNLOAD DAN EKSTRAKSI SEMUA CV


cv_file_id_list = []
cv_download_path_list = []
cv_text_path_list = []
cv_text_list = []
cv_text_length_list = []
cv_extraction_status_list = []
cv_extraction_note_list = []

print("=== MULAI DOWNLOAD DAN EKSTRAKSI CV ===")

for idx, row in cleaned_df.iterrows():
    no = row.get("no", idx + 1)
    nama = row.get("nama_lengkap", f"kandidat_{no}")
    link_cv_raw = row.get("link_cv", "")

    candidate_name_safe = safe_filename(nama)

    cv_file_id = ""
    cv_download_path = ""
    cv_text_path = ""
    cv_text = ""
    cv_text_length = 0
    extraction_status = ""
    extraction_note = ""

    # ----------------------------
    # 1. Bersihkan link dan ambil file ID
    # ----------------------------
    clean_cv_url = extract_url_from_cell(link_cv_raw)
    cv_file_id = extract_google_drive_file_id(clean_cv_url)

    if is_blank(clean_cv_url):
        extraction_status = "FAILED"
        extraction_note = "Missing CV link"
        cleaned_df.loc[idx, "flag"] = append_flag_to_text(cleaned_df.loc[idx, "flag"], "MISSING_CV")

    elif cv_file_id == "":
        extraction_status = "FAILED"
        extraction_note = "Google Drive file ID not found"
        cleaned_df.loc[idx, "flag"] = append_flag_to_text(cleaned_df.loc[idx, "flag"], "INVALID_CV_LINK")

    else:
        # ----------------------------
        # 2. Download PDF
        # ----------------------------
        pdf_filename = f"{int(no):03d}_{candidate_name_safe}.pdf"
        output_pdf_path = CV_DOWNLOAD_DIR / pdf_filename

        download_success, download_note = download_cv_from_drive(
            file_id=cv_file_id,
            output_path=output_pdf_path
        )

        if not download_success:
            extraction_status = "FAILED"
            extraction_note = f"Download failed: {download_note}"
            cleaned_df.loc[idx, "flag"] = append_flag_to_text(cleaned_df.loc[idx, "flag"], "CV_DOWNLOAD_FAILED")

        else:
            cv_download_path = str(output_pdf_path)

            # ----------------------------
            # 3. Ekstraksi teks PDF
            # ----------------------------
            extracted_text, extract_note = extract_text_from_pdf(output_pdf_path)
            cv_text = extracted_text
            cv_text_length = len(cv_text)

            text_filename = f"{int(no):03d}_{candidate_name_safe}.txt"
            output_text_path = CV_TEXT_DIR / text_filename

            try:
                with open(output_text_path, "w", encoding="utf-8") as f:
                    f.write(cv_text)

                cv_text_path = str(output_text_path)

            except Exception as e:
                cv_text_path = ""
                extraction_note = f"Text file save failed: {e}"

            if cv_text_length == 0:
                extraction_status = "FAILED"
                extraction_note = f"Extraction failed or empty text: {extract_note}"
                cleaned_df.loc[idx, "flag"] = append_flag_to_text(cleaned_df.loc[idx, "flag"], "CV_EXTRACTION_FAILED")

            elif cv_text_length < MIN_CV_TEXT_LENGTH:
                extraction_status = "TEXT_TOO_SHORT"
                extraction_note = f"Extracted text too short: {cv_text_length} characters"
                cleaned_df.loc[idx, "flag"] = append_flag_to_text(cleaned_df.loc[idx, "flag"], "TEXT_TOO_SHORT")

            else:
                extraction_status = "SUCCESS"
                extraction_note = "CV extracted successfully"

    # ----------------------------
    # Simpan hasil per kandidat
    # ----------------------------
    cv_file_id_list.append(cv_file_id)
    cv_download_path_list.append(cv_download_path)
    cv_text_path_list.append(cv_text_path)
    cv_text_list.append(cv_text)
    cv_text_length_list.append(cv_text_length)
    cv_extraction_status_list.append(extraction_status)
    cv_extraction_note_list.append(extraction_note)

    print(f"{int(no):03d}. {nama} → {extraction_status} | {extraction_note}")

print("=== SELESAI DOWNLOAD DAN EKSTRAKSI CV ===")

=== MULAI DOWNLOAD DAN EKSTRAKSI CV ===
001. Tomi Wijaya → FAILED | Google Drive file ID not found
002. Regina Meidika → SUCCESS | CV extracted successfully
003. Angga Ian Saputra → SUCCESS | CV extracted successfully
004. Malesta Witaka Kussumasari → FAILED | Google Drive file ID not found
005. Ine Safitri Akes Parema → SUCCESS | CV extracted successfully
006. Muhammad Rafli Choirul Umam → FAILED | Google Drive file ID not found
007. Nicholas Noverhino Ama Payong → SUCCESS | CV extracted successfully
008. Nindya Meiannur Shahnaz Purnomo → SUCCESS | CV extracted successfully
009. Suhairi Ibnu Abdillah → FAILED | Google Drive file ID not found
010. Matthew Arthur Christopher Nathanael Hutapea → FAILED | Download failed: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to acce

In [ ]:

# SIMPAN HASIL EKSTRAKSI KE CLEANED_DF


cleaned_df["cv_file_id"] = cv_file_id_list
cleaned_df["cv_download_path"] = cv_download_path_list
cleaned_df["cv_text_path"] = cv_text_path_list
cleaned_df["cv_text"] = cv_text_list
cleaned_df["cv_text_length"] = cv_text_length_list
cleaned_df["cv_extraction_status"] = cv_extraction_status_list
cleaned_df["cv_extraction_note"] = cv_extraction_note_list

print(" Hasil ekstraksi CV berhasil disimpan ke cleaned_df.")

display(
    cleaned_df[
        [
            "no",
            "nama_lengkap",
            "link_cv",
            "cv_file_id",
            "cv_text_length",
            "cv_extraction_status",
            "cv_extraction_note",
            "flag"
        ]
    ].head(10)
)

 Hasil ekstraksi CV berhasil disimpan ke cleaned_df.


,no,nama_lengkap,link_cv,cv_file_id,cv_text_length,cv_extraction_status,cv_extraction_note,flag
0,1,Tomi Wijaya,https://drive.google.com/drive/folders/1NT1u91...,,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
1,2,Regina Meidika,https://drive.google.com/file/d/1vjWWEJFciU3yR...,1vjWWEJFciU3yR0CSTcMHbVkmASYGHSa4,4305,SUCCESS,CV extracted successfully,
2,3,Angga Ian Saputra,https://drive.google.com/file/d/1yZkfmyV3MZDaL...,1yZkfmyV3MZDaLVmEOD2d_9xCnO_0jfwo,2461,SUCCESS,CV extracted successfully,
3,4,Malesta Witaka Kussumasari,https://drive.google.com/drive/folders/1fIQTMF...,,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
4,5,Ine Safitri Akes Parema,https://drive.google.com/file/d/1HcZNURRqoIA7j...,1HcZNURRqoIA7jGK_aX8OLnulnEBXLqNB,2688,SUCCESS,CV extracted successfully,
5,6,Muhammad Rafli Choirul Umam,https://drive.google.com/drive/folders/1J8rfhs...,,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
6,7,Nicholas Noverhino Ama Payong,https://drive.google.com/file/d/1JVkZzAvBpfimQ...,1JVkZzAvBpfimQzLvhd_zMyGtNy2b-G-N,4271,SUCCESS,CV extracted successfully,
7,8,Nindya Meiannur Shahnaz Purnomo,https://drive.google.com/file/d/1IbqkCxeH3IiwB...,1IbqkCxeH3IiwBK4TIn8BRTIMiEw13V7g,3785,SUCCESS,CV extracted successfully,
8,9,Suhairi Ibnu Abdillah,https://drive.google.com/drive/folders/11Rb8DP...,,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
9,10,Matthew Arthur Christopher Nathanael Hutapea,https://drive.google.com/file/d/1rWylbBy2LBxJe...,1rWylbBy2LBxJe0Lr9jyS9hYcrQ7vRnYy,0,FAILED,Download failed: Failed to retrieve file url:\...,CV_DOWNLOAD_FAILED


In [ ]:

# UPDATE STATUS ADMINISTRASI SETELAH EKSTRAKSI CV


def update_status_after_cv_extraction(row):
    status = row.get("status_administrasi", "")
    flag = row.get("flag", "")

    cv_problem_flags = [
        "CV_DOWNLOAD_FAILED",
        "CV_EXTRACTION_FAILED",
        "TEXT_TOO_SHORT"
    ]

    has_cv_problem = any(problem_flag in flag for problem_flag in cv_problem_flags)

    if has_cv_problem:
        if status == "Administrasi Lengkap":
            return "Perlu Dicek"

    return status


cleaned_df["status_administrasi"] = cleaned_df.apply(
    update_status_after_cv_extraction,
    axis=1
)

# Update juga ke output utama
screening_output_df["Status Administrasi"] = cleaned_df["status_administrasi"]
screening_output_df["Flag"] = cleaned_df["flag"]

print(" Status Administrasi dan Flag diperbarui setelah ekstraksi CV.")

display(
    screening_output_df[
        [
            "No",
            "Nama Lengkap",
            "Status Administrasi",
            "Flag"
        ]
    ].head(10)
)

 Status Administrasi dan Flag diperbarui setelah ekstraksi CV.


,No,Nama Lengkap,Status Administrasi,Flag
0,1,Tomi Wijaya,Perlu Dicek,CV_LINK_IS_FOLDER; INVALID_CV_LINK
1,2,Regina Meidika,Administrasi Lengkap,
2,3,Angga Ian Saputra,Administrasi Lengkap,
3,4,Malesta Witaka Kussumasari,Perlu Dicek,CV_LINK_IS_FOLDER; INVALID_CV_LINK
4,5,Ine Safitri Akes Parema,Administrasi Lengkap,
5,6,Muhammad Rafli Choirul Umam,Perlu Dicek,CV_LINK_IS_FOLDER; INVALID_CV_LINK
6,7,Nicholas Noverhino Ama Payong,Administrasi Lengkap,
7,8,Nindya Meiannur Shahnaz Purnomo,Administrasi Lengkap,
8,9,Suhairi Ibnu Abdillah,Perlu Dicek,CV_LINK_IS_FOLDER; INVALID_CV_LINK
9,10,Matthew Arthur Christopher Nathanael Hutapea,Perlu Dicek,CV_DOWNLOAD_FAILED


In [ ]:

# RINGKASAN HASIL EKSTRAKSI CV


print("=== RINGKASAN STATUS EKSTRAKSI CV ===")
display(cleaned_df["cv_extraction_status"].value_counts(dropna=False))

print("\n=== RINGKASAN PANJANG TEKS CV ===")
display(cleaned_df["cv_text_length"].describe())

print("\n=== KANDIDAT DENGAN MASALAH EKSTRAKSI CV ===")

cv_problem_df = cleaned_df[
    cleaned_df["cv_extraction_status"].isin([
        "FAILED",
        "TEXT_TOO_SHORT"
    ])
]

if len(cv_problem_df) > 0:
    display(
        cv_problem_df[
            [
                "no",
                "nama_lengkap",
                "link_cv",
                "cv_text_length",
                "cv_extraction_status",
                "cv_extraction_note",
                "flag"
            ]
        ]
    )
else:
    print(" Tidak ada masalah ekstraksi CV.")

=== RINGKASAN STATUS EKSTRAKSI CV ===


,count
cv_extraction_status,
SUCCESS,197
FAILED,74



=== RINGKASAN PANJANG TEKS CV ===


,cv_text_length
count,271.000000
mean,3098.977860
std,3000.599933
min,0.000000
25%,0.000000
50%,2718.000000
75%,4454.500000
max,16326.000000



=== KANDIDAT DENGAN MASALAH EKSTRAKSI CV ===


,no,nama_lengkap,link_cv,cv_text_length,cv_extraction_status,cv_extraction_note,flag
0,1,Tomi Wijaya,https://drive.google.com/drive/folders/1NT1u91...,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
3,4,Malesta Witaka Kussumasari,https://drive.google.com/drive/folders/1fIQTMF...,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
5,6,Muhammad Rafli Choirul Umam,https://drive.google.com/drive/folders/1J8rfhs...,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
8,9,Suhairi Ibnu Abdillah,https://drive.google.com/drive/folders/11Rb8DP...,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
9,10,Matthew Arthur Christopher Nathanael Hutapea,https://drive.google.com/file/d/1rWylbBy2LBxJe...,0,FAILED,Download failed: Failed to retrieve file url:\...,CV_DOWNLOAD_FAILED
...,...,...,...,...,...,...,...
260,261,Anggun Afreza Putri,https://drive.google.com/drive/folders/1VPOg2g...,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
261,262,SUHAIMAH,https://drive.google.com/drive/u/0/folders/1EV...,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
264,265,Ade Puspita,https://drive.google.com/drive/folders/153jdJM...,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK
266,267,Nasywa Giovanni Maida,https://drive.google.com/drive/folders/1iEFXjN...,0,FAILED,Google Drive file ID not found,CV_LINK_IS_FOLDER; INVALID_CV_LINK


In [ ]:

# UPDATE CANDIDATE MATCHING TEXT DENGAN ISI CV


def build_candidate_text_with_cv(row):
    """
    Menggabungkan data Form Responses dan isi CV untuk matching.
    """
    text_parts = [
        row.get("pendidikan", ""),
        row.get("jurusan_angkatan", ""),
        row.get("asal_instansi", ""),
        row.get("alasan_memilih_divisi", ""),
        row.get("pilihan_divisi_pertama", ""),
        row.get("pilihan_divisi_kedua", ""),
        row.get("cv_text", "")
    ]

    combined_text = " ".join([clean_text(x) for x in text_parts])
    return combined_text


cleaned_df["candidate_matching_text"] = cleaned_df.apply(
    build_candidate_text_with_cv,
    axis=1
)

print(" candidate_matching_text berhasil diperbarui dengan isi CV.")

display(
    cleaned_df[
        [
            "nama_lengkap",
            "cv_text_length",
            "cv_extraction_status",
            "candidate_matching_text"
        ]
    ].head()
)

 candidate_matching_text berhasil diperbarui dengan isi CV.


,nama_lengkap,cv_text_length,cv_extraction_status,candidate_matching_text
0,Tomi Wijaya,0,FAILED,Mahasiswa Manajemen - 2022 Universitas Bhayang...
1,Regina Meidika,4305,SUCCESS,Fresh Graduate Hukum - 2026 Universitas Lampun...
2,Angga Ian Saputra,2461,SUCCESS,Fresh Graduate Manajemen - 2021 Universitas Bh...
3,Malesta Witaka Kussumasari,0,FAILED,Mahasiswa Akuntansi - 2024 UPN Veteran Yogyaka...
4,Ine Safitri Akes Parema,2688,SUCCESS,Mahasiswa Psikologi - 2023 Universitas Negeri ...


In [ ]:

# RINGKASAN BAGIAN 5


print("=== RINGKASAN BAGIAN 5 ===")
print(f"Jumlah kandidat diproses              : {len(cleaned_df)}")
print(f"Jumlah CV berhasil diekstrak          : {(cleaned_df['cv_extraction_status'] == 'SUCCESS').sum()}")
print(f"Jumlah CV gagal / perlu dicek         : {(cleaned_df['cv_extraction_status'] != 'SUCCESS').sum()}")
print(f"Folder download CV                    : {CV_DOWNLOAD_DIR}")
print(f"Folder hasil teks CV                  : {CV_TEXT_DIR}")

print("\nKolom baru yang ditambahkan:")
new_cv_columns = [
    "cv_file_id",
    "cv_download_path",
    "cv_text_path",
    "cv_text",
    "cv_text_length",
    "cv_extraction_status",
    "cv_extraction_note",
    "candidate_matching_text"
]

for col in new_cv_columns:
    print(f"- {col}")

print("\n Bagian 5 selesai.")
print("Lanjut ke Bagian 6 — Matching Matrix dan Scoring Awal.")

=== RINGKASAN BAGIAN 5 ===
Jumlah kandidat diproses              : 271
Jumlah CV berhasil diekstrak          : 197
Jumlah CV gagal / perlu dicek         : 74
Folder download CV                    : /content/drive/Shareddrives/CV Screening Automation/04_temp_cv_downloads
Folder hasil teks CV                  : /content/drive/Shareddrives/CV Screening Automation/05_cv_text

Kolom baru yang ditambahkan:
- cv_file_id
- cv_download_path
- cv_text_path
- cv_text
- cv_text_length
- cv_extraction_status
- cv_extraction_note
- candidate_matching_text

 Bagian 5 selesai.
Lanjut ke Bagian 6 — Matching Matrix dan Scoring Awal.


In [ ]:
# BAGIAN 6 — MATCHING MATRIX DAN SCORING AWAL


import re
import pandas as pd
import numpy as np

In [ ]:

# VALIDASI DEPENDENCY BAGIAN SEBELUMNYA


required_variables = [
    "cleaned_df",
    "skill_df",
    "academic_df",
    "division_keywords_df"
]

missing_variables = []

for var_name in required_variables:
    if var_name not in globals():
        missing_variables.append(var_name)

if len(missing_variables) > 0:
    raise ValueError(
        "Variabel berikut belum tersedia. Pastikan Bagian 1-5 sudah dijalankan: "
        + ", ".join(missing_variables)
    )

# Pastikan fungsi clean_text dan is_blank tersedia.
# Jika belum tersedia karena cell sebelumnya belum dijalankan, buat fallback sederhana.
if "clean_text" not in globals():
    def clean_text(value):
        if pd.isna(value):
            return ""
        value = str(value)
        value = value.replace("\n", " ")
        value = re.sub(r"\s+", " ", value)
        value = value.strip()
        if value.lower() in ["nan", "none", "null"]:
            return ""
        return value

if "is_blank" not in globals():
    def is_blank(value):
        value = clean_text(value)
        return value == "" or value.lower() in ["-", "tidak ada", "none", "null", "nan", "n/a"]

if "append_flag_to_text" not in globals():
    def append_flag_to_text(existing_flag_text, new_flag):
        existing_flag_text = clean_text(existing_flag_text)

        if existing_flag_text == "":
            return new_flag

        flags = [flag.strip() for flag in existing_flag_text.split(";") if flag.strip()]

        if new_flag not in flags:
            flags.append(new_flag)

        return "; ".join(flags)

print(" Dependency Bagian 6 sudah siap.")

 Dependency Bagian 6 sudah siap.


In [ ]:

# PASTIKAN CANDIDATE MATCHING TEXT TERSEDIA


def build_candidate_text_fallback(row):
    """
    Fallback jika candidate_matching_text belum tersedia.
    Idealnya Bagian 5 sudah menambahkan cv_text ke candidate_matching_text.
    """
    text_parts = [
        row.get("pendidikan", ""),
        row.get("jurusan_angkatan", ""),
        row.get("asal_instansi", ""),
        row.get("alasan_memilih_divisi", ""),
        row.get("pilihan_divisi_pertama", ""),
        row.get("pilihan_divisi_kedua", ""),
        row.get("cv_text", "")
    ]

    return " ".join([clean_text(x) for x in text_parts])


if "candidate_matching_text" not in cleaned_df.columns:
    print(" candidate_matching_text belum ditemukan. Membuat fallback dari data kandidat.")
    cleaned_df["candidate_matching_text"] = cleaned_df.apply(
        build_candidate_text_fallback,
        axis=1
    )
else:
    print(" candidate_matching_text ditemukan.")

display(
    cleaned_df[
        [
            "nama_lengkap",
            "pilihan_divisi_pertama",
            "pilihan_divisi_kedua",
            "candidate_matching_text"
        ]
    ].head()
)

 candidate_matching_text ditemukan.


,nama_lengkap,pilihan_divisi_pertama,pilihan_divisi_kedua,candidate_matching_text
0,Tomi Wijaya,HR Organizational Development,Business Development,Mahasiswa Manajemen - 2022 Universitas Bhayang...
1,Regina Meidika,HR Organizational Development,HR Talent Acquisition,Fresh Graduate Hukum - 2026 Universitas Lampun...
2,Angga Ian Saputra,HR Talent Acquisition,HR Talent Acquisition,Fresh Graduate Manajemen - 2021 Universitas Bh...
3,Malesta Witaka Kussumasari,Data Analyst,Data Analyst,Mahasiswa Akuntansi - 2024 UPN Veteran Yogyaka...
4,Ine Safitri Akes Parema,HR Talent Acquisition,HR Organizational Development,Mahasiswa Psikologi - 2023 Universitas Negeri ...


In [ ]:

# FUNGSI NORMALISASI DAN MATCHING KEYWORD


def normalize_for_matching(text):
    """
    Normalisasi teks untuk matching:
    - lowercase
    - hapus newline
    - rapikan spasi
    """
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


def keyword_to_pattern(keyword):
    """
    Membuat regex pattern untuk keyword.
    Untuk keyword pendek seperti R, AI, HR, SQL:
    gunakan word boundary agar tidak match di tengah kata.
    Untuk keyword panjang:
    tetap gunakan boundary longgar agar frasa bisa terdeteksi.
    """
    keyword_norm = normalize_for_matching(keyword)

    if keyword_norm == "":
        return None

    escaped = re.escape(keyword_norm)

    # Jika keyword sangat pendek, wajib exact token agar tidak terlalu banyak false positive.
    if len(keyword_norm) <= 2:
        return r"(?<![a-zA-Z0-9])" + escaped + r"(?![a-zA-Z0-9])"

    # Untuk keyword normal, gunakan boundary agar tidak match sebagai potongan kata.
    return r"(?<![a-zA-Z0-9])" + escaped + r"(?![a-zA-Z0-9])"


def contains_keyword(text, keyword):
    """
    Mengecek apakah keyword muncul pada text.
    Menggunakan regex sederhana agar lebih aman dari false positive.
    """
    text_norm = normalize_for_matching(text)
    pattern = keyword_to_pattern(keyword)

    if pattern is None:
        return False

    return re.search(pattern, text_norm) is not None

In [ ]:

# BERSIHKAN MATRIX


def clean_matrix_text_columns(df, text_columns):
    """
    Membersihkan kolom teks pada matrix.
    """
    df = df.copy()

    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)

    return df


skill_df_clean = clean_matrix_text_columns(
    skill_df,
    ["division", "skill_keyword"]
)

academic_df_clean = clean_matrix_text_columns(
    academic_df,
    ["division", "major_keyword"]
)

division_keywords_df_clean = clean_matrix_text_columns(
    division_keywords_df,
    ["division", "keyword"]
)

# Hapus baris kosong
skill_df_clean = skill_df_clean[
    (~skill_df_clean["division"].apply(is_blank)) &
    (~skill_df_clean["skill_keyword"].apply(is_blank))
].copy()

academic_df_clean = academic_df_clean[
    (~academic_df_clean["division"].apply(is_blank)) &
    (~academic_df_clean["major_keyword"].apply(is_blank))
].copy()

division_keywords_df_clean = division_keywords_df_clean[
    (~division_keywords_df_clean["division"].apply(is_blank)) &
    (~division_keywords_df_clean["keyword"].apply(is_blank))
].copy()

# Pastikan score academic numerik
academic_df_clean["score"] = pd.to_numeric(
    academic_df_clean["score"],
    errors="coerce"
).fillna(1)

print(" Matrix berhasil dibersihkan.")
print(f"Skill matrix rows          : {len(skill_df_clean)}")
print(f"Academic matrix rows       : {len(academic_df_clean)}")
print(f"Division keywords rows     : {len(division_keywords_df_clean)}")

 Matrix berhasil dibersihkan.
Skill matrix rows          : 1021
Academic matrix rows       : 466
Division keywords rows     : 864


In [ ]:

# CEK KESESUAIAN NAMA DIVISI FORM DENGAN MATRIX


def get_unique_clean_values(series):
    return sorted(
        series.dropna()
        .astype(str)
        .map(clean_text)
        .replace("", np.nan)
        .dropna()
        .unique()
    )


form_divisions = sorted(
    set(get_unique_clean_values(cleaned_df["pilihan_divisi_pertama"])) |
    set(get_unique_clean_values(cleaned_df["pilihan_divisi_kedua"]))
)

skill_divisions = set(get_unique_clean_values(skill_df_clean["division"]))
academic_divisions = set(get_unique_clean_values(academic_df_clean["division"]))
keyword_divisions = set(get_unique_clean_values(division_keywords_df_clean["division"]))

print("=== CEK DIVISI FORM VS MATRIX ===")

for matrix_name, matrix_divisions in [
    ("Skill Matrix", skill_divisions),
    ("Academic Matrix", academic_divisions),
    ("Division Keywords", keyword_divisions)
]:
    missing = sorted(set(form_divisions) - matrix_divisions)

    print(f"\n{matrix_name}:")
    if len(missing) == 0:
        print(" Semua divisi dari Form Responses ditemukan di matrix.")
    else:
        print(" Ada divisi dari Form Responses yang belum ada di matrix:")
        for div in missing:
            print(f"- {div}")

=== CEK DIVISI FORM VS MATRIX ===

Skill Matrix:
 Semua divisi dari Form Responses ditemukan di matrix.

Academic Matrix:
 Semua divisi dari Form Responses ditemukan di matrix.

Division Keywords:
 Semua divisi dari Form Responses ditemukan di matrix.


In [ ]:

# FUNGSI AMBIL MATRIX PER DIVISI


def get_skills_for_division(division):
    division = clean_text(division)

    subset = skill_df_clean[
        skill_df_clean["division"].str.lower() == division.lower()
    ]

    return subset["skill_keyword"].dropna().astype(str).tolist()


def get_academic_rows_for_division(division):
    division = clean_text(division)

    subset = academic_df_clean[
        academic_df_clean["division"].str.lower() == division.lower()
    ]

    return subset.copy()


def get_keywords_for_division(division):
    division = clean_text(division)

    subset = division_keywords_df_clean[
        division_keywords_df_clean["division"].str.lower() == division.lower()
    ]

    return subset["keyword"].dropna().astype(str).tolist()

In [ ]:

# SKILL MATCHING


def match_skills(candidate_text, division):
    """
    Mencari skill keyword divisi yang muncul pada teks kandidat.
    """
    skills = get_skills_for_division(division)
    matched_skills = []

    for skill in skills:
        if contains_keyword(candidate_text, skill):
            matched_skills.append(skill)

    matched_skills = sorted(set(matched_skills), key=lambda x: x.lower())
    return matched_skills


def calculate_skill_score(matched_skills):
    """
    Skill Score:
    0-1 match = 1
    2 match   = 2
    3 match   = 3
    4 match   = 4
    5+ match  = 5
    """
    match_count = len(matched_skills)

    if match_count <= 1:
        return 1
    elif match_count == 2:
        return 2
    elif match_count == 3:
        return 3
    elif match_count == 4:
        return 4
    else:
        return 5

In [ ]:

# ACADEMIC MATCHING


def build_academic_text(row):
    """
    Sumber teks untuk academic matching.
    Jangan memakai seluruh CV agar tidak terlalu banyak false positive.
    """
    text_parts = [
        row.get("pendidikan", ""),
        row.get("jurusan_angkatan", "")
    ]

    return " ".join([clean_text(x) for x in text_parts])


def match_academic(row, division):
    """
    Mencocokkan jurusan kandidat dengan academic matrix.
    Mengembalikan:
    - matched_major list
    - academic_score
    """
    academic_text = build_academic_text(row)
    academic_rows = get_academic_rows_for_division(division)

    matched = []

    for _, academic_row in academic_rows.iterrows():
        major_keyword = academic_row.get("major_keyword", "")
        score = academic_row.get("score", 1)

        if contains_keyword(academic_text, major_keyword):
            matched.append({
                "major_keyword": major_keyword,
                "score": float(score)
            })

    if len(matched) == 0:
        return [], 1

    best_score = max(item["score"] for item in matched)

    matched_major = [
        item["major_keyword"]
        for item in matched
        if item["score"] == best_score
    ]

    matched_major = sorted(set(matched_major), key=lambda x: x.lower())

    return matched_major, best_score

In [ ]:

# RELATEDNESS MATCHING


def match_division_keywords(candidate_text, division):
    """
    Mencari keyword pengalaman/jobdesc divisi yang muncul pada teks kandidat.
    """
    keywords = get_keywords_for_division(division)
    matched_keywords = []

    for keyword in keywords:
        if contains_keyword(candidate_text, keyword):
            matched_keywords.append(keyword)

    matched_keywords = sorted(set(matched_keywords), key=lambda x: x.lower())
    return matched_keywords


def detect_duration_signal(text):
    """
    Deteksi sederhana apakah ada indikasi durasi pengalaman.
    Ini hanya sebagai bonus untuk relatedness.
    """
    text_norm = normalize_for_matching(text)

    duration_patterns = [
        r"\b\d+\s*bulan\b",
        r"\b\d+\s*month[s]?\b",
        r"\b\d+\s*tahun\b",
        r"\b\d+\s*year[s]?\b",
        r"\bjan[uary]*\s*\d{4}\s*[-–]\s*\w+\s*\d{4}\b",
        r"\b\d{4}\s*[-–]\s*\d{4}\b"
    ]

    for pattern in duration_patterns:
        if re.search(pattern, text_norm):
            return True

    return False


def calculate_relatedness_score(matched_keywords, candidate_text):
    """
    Relatedness Score:
    0 match   = 1
    1-2 match = 2
    3-4 match = 3
    5+ match  = 4
    5+ match + indikasi durasi = 5

    Catatan:
    Durasi hanya menjadi bonus, bukan syarat utama.
    """
    match_count = len(matched_keywords)
    has_duration = detect_duration_signal(candidate_text)

    if match_count == 0:
        return 1
    elif match_count <= 2:
        return 2
    elif match_count <= 4:
        return 3
    else:
        if has_duration:
            return 5
        return 4

In [ ]:
# FUNGSI MATCHING UNTUK SATU DIVISI


def score_candidate_for_division(row, division):
    """
    Menghitung skill, academic, dan relatedness untuk satu kandidat terhadap satu divisi.
    """
    candidate_text = row.get("candidate_matching_text", "")

    # Skill
    matched_skills = match_skills(candidate_text, division)
    skill_score = calculate_skill_score(matched_skills)

    # Academic
    matched_academic, academic_score = match_academic(row, division)

    # Relatedness
    matched_keywords = match_division_keywords(candidate_text, division)
    relatedness_score = calculate_relatedness_score(
        matched_keywords,
        candidate_text
    )

    result = {
        "matched_skills": "; ".join(matched_skills),
        "matched_academic": "; ".join(matched_academic),
        "matched_keywords": "; ".join(matched_keywords),
        "skill_score": skill_score,
        "academic_score": academic_score,
        "relatedness_score": relatedness_score
    }

    return result

In [ ]:
# JALANKAN MATCHING UNTUK SEMUA KANDIDAT


results = []

for idx, row in cleaned_df.iterrows():
    divisi_pertama = row.get("pilihan_divisi_pertama", "")
    divisi_kedua = row.get("pilihan_divisi_kedua", "")

    # Jika divisi tidak ditemukan di matrix, tambahkan flag.
    all_matrix_divisions = skill_divisions & academic_divisions & keyword_divisions

    if divisi_pertama not in all_matrix_divisions:
        cleaned_df.loc[idx, "flag"] = append_flag_to_text(
            cleaned_df.loc[idx, "flag"],
            "DIVISION_1_NOT_FOUND_IN_MATRIX"
        )

    if divisi_kedua not in all_matrix_divisions:
        cleaned_df.loc[idx, "flag"] = append_flag_to_text(
            cleaned_df.loc[idx, "flag"],
            "DIVISION_2_NOT_FOUND_IN_MATRIX"
        )

    # Score pilihan divisi pertama
    score_1 = score_candidate_for_division(row, divisi_pertama)

    # Score pilihan divisi kedua
    score_2 = score_candidate_for_division(row, divisi_kedua)

    results.append({
        "matched_skills_divisi_pertama": score_1["matched_skills"],
        "matched_academic_divisi_pertama": score_1["matched_academic"],
        "matched_keywords_divisi_pertama": score_1["matched_keywords"],
        "skill_score_divisi_pertama": score_1["skill_score"],
        "academic_score_divisi_pertama": score_1["academic_score"],
        "relatedness_score_divisi_pertama": score_1["relatedness_score"],

        "matched_skills_divisi_kedua": score_2["matched_skills"],
        "matched_academic_divisi_kedua": score_2["matched_academic"],
        "matched_keywords_divisi_kedua": score_2["matched_keywords"],
        "skill_score_divisi_kedua": score_2["skill_score"],
        "academic_score_divisi_kedua": score_2["academic_score"],
        "relatedness_score_divisi_kedua": score_2["relatedness_score"]
    })

matching_result_df = pd.DataFrame(results)

# Gabungkan ke cleaned_df
for col in matching_result_df.columns:
    cleaned_df[col] = matching_result_df[col]

print(" Matching matrix dan scoring awal selesai.")

 Matching matrix dan scoring awal selesai.


In [ ]:
# PREVIEW HASIL MATCHING


preview_columns = [
    "nama_lengkap",
    "pilihan_divisi_pertama",
    "matched_skills_divisi_pertama",
    "matched_academic_divisi_pertama",
    "matched_keywords_divisi_pertama",
    "skill_score_divisi_pertama",
    "academic_score_divisi_pertama",
    "relatedness_score_divisi_pertama",
    "pilihan_divisi_kedua",
    "matched_skills_divisi_kedua",
    "matched_academic_divisi_kedua",
    "matched_keywords_divisi_kedua",
    "skill_score_divisi_kedua",
    "academic_score_divisi_kedua",
    "relatedness_score_divisi_kedua",
    "flag"
]

display(cleaned_df[preview_columns].head(10))

,nama_lengkap,pilihan_divisi_pertama,matched_skills_divisi_pertama,matched_academic_divisi_pertama,matched_keywords_divisi_pertama,skill_score_divisi_pertama,academic_score_divisi_pertama,relatedness_score_divisi_pertama,pilihan_divisi_kedua,matched_skills_divisi_kedua,matched_academic_divisi_kedua,matched_keywords_divisi_kedua,skill_score_divisi_kedua,academic_score_divisi_kedua,relatedness_score_divisi_kedua,flag
0,Tomi Wijaya,HR Organizational Development,organizational development,Manajemen,organizational development,1,5.0,2,Business Development,business development,Manajemen,business development,1,5.0,2,CV_LINK_IS_FOLDER; INVALID_CV_LINK
1,Regina Meidika,HR Organizational Development,Google Workspace; komunikasi; koordinasi; Micr...,Hukum,organizational development; pengelolaan dokume...,5,4.0,3,HR Talent Acquisition,Google Workspace; komunikasi; koordinasi; Micr...,Hukum,proses rekrutmen; talent acquisition; wawancara,5,4.0,3,
2,Angga Ian Saputra,HR Talent Acquisition,Google Workspace; komunikasi; Microsoft Office...,Manajemen,proses rekrutmen; talent acquisition,5,5.0,2,HR Talent Acquisition,Google Workspace; komunikasi; Microsoft Office...,Manajemen,proses rekrutmen; talent acquisition,5,5.0,2,
3,Malesta Witaka Kussumasari,Data Analyst,analisis data; pengolahan data,Akuntansi,analisis data; menganalisis data; pengolahan data,2,4.0,3,Data Analyst,analisis data; pengolahan data,Akuntansi,analisis data; menganalisis data; pengolahan data,2,4.0,3,CV_LINK_IS_FOLDER; INVALID_CV_LINK
4,Ine Safitri Akes Parema,HR Talent Acquisition,communication; employee engagement; employer b...,Psikologi,employee engagement; employer branding; offeri...,5,5.0,5,HR Organizational Development,communication; employee engagement; Google Wor...,Psikologi,evaluasi organisasi; organizational developmen...,5,5.0,5,
5,Muhammad Rafli Choirul Umam,Data Analyst,,,,1,1.0,1,Data Analyst,,,,1,1.0,1,CV_LINK_IS_FOLDER; INVALID_CV_LINK
6,Nicholas Noverhino Ama Payong,Data Analyst,analisis data; dashboard; data analysis; datab...,Informatika; Teknik Informatika,analisis data; dashboard; data analysis; datab...,5,5.0,5,Data Analyst,analisis data; dashboard; data analysis; datab...,Informatika; Teknik Informatika,analisis data; dashboard; data analysis; datab...,5,5.0,5,
7,Nindya Meiannur Shahnaz Purnomo,Data Analyst,analytical thinking; attention to detail; crit...,Statistika,data analysis; data visualization; database; r...,5,5.0,3,Business Development,business development; communication; komunikas...,,business development; partnership; pengembanga...,5,1.0,3,
8,Suhairi Ibnu Abdillah,Graphic Design,graphic design,,graphic design,1,1.0,2,Content Creator,content creator,,,1,1.0,1,CV_LINK_IS_FOLDER; INVALID_CV_LINK
9,Matthew Arthur Christopher Nathanael Hutapea,Customer Services,customer service; komunikasi; problem solving,,customer service; ramah,3,1.0,2,Event Project,organisasi; problem solving,,organisasi,2,1.0,2,CV_DOWNLOAD_FAILED


In [ ]:
# RINGKASAN SKOR MATCHING


print("=== RINGKASAN SKILL SCORE DIVISI PERTAMA ===")
display(cleaned_df["skill_score_divisi_pertama"].value_counts().sort_index())

print("=== RINGKASAN ACADEMIC SCORE DIVISI PERTAMA ===")
display(cleaned_df["academic_score_divisi_pertama"].value_counts().sort_index())

print("=== RINGKASAN RELATEDNESS SCORE DIVISI PERTAMA ===")
display(cleaned_df["relatedness_score_divisi_pertama"].value_counts().sort_index())

print("=== RINGKASAN SKILL SCORE DIVISI KEDUA ===")
display(cleaned_df["skill_score_divisi_kedua"].value_counts().sort_index())

print("=== RINGKASAN ACADEMIC SCORE DIVISI KEDUA ===")
display(cleaned_df["academic_score_divisi_kedua"].value_counts().sort_index())

print("=== RINGKASAN RELATEDNESS SCORE DIVISI KEDUA ===")
display(cleaned_df["relatedness_score_divisi_kedua"].value_counts().sort_index())

=== RINGKASAN SKILL SCORE DIVISI PERTAMA ===


,count
skill_score_divisi_pertama,
1,23
2,13
3,21
4,18
5,196


=== RINGKASAN ACADEMIC SCORE DIVISI PERTAMA ===


,count
academic_score_divisi_pertama,
1.0,64
4.0,57
5.0,150


=== RINGKASAN RELATEDNESS SCORE DIVISI PERTAMA ===


...

,count
skill_score_divisi_kedua,
1,21
2,23
3,22
4,17
5,188


=== RINGKASAN ACADEMIC SCORE DIVISI KEDUA ===


,count
academic_score_divisi_kedua,
1.0,110
4.0,53
5.0,108


=== RINGKASAN RELATEDNESS SCORE DIVISI KEDUA ===


,count
relatedness_score_divisi_kedua,
1,21
2,104
3,80
4,28
5,38


In [ ]:
# CEK MATCHING KOSONG


no_skill_match_1 = cleaned_df[
    cleaned_df["matched_skills_divisi_pertama"].apply(is_blank)
]

no_keyword_match_1 = cleaned_df[
    cleaned_df["matched_keywords_divisi_pertama"].apply(is_blank)
]

no_academic_match_1 = cleaned_df[
    cleaned_df["matched_academic_divisi_pertama"].apply(is_blank)
]

print("=== CEK MATCHING KOSONG UNTUK PILIHAN DIVISI PERTAMA ===")
print(f"Tanpa skill match       : {len(no_skill_match_1)} kandidat")
print(f"Tanpa keyword match     : {len(no_keyword_match_1)} kandidat")
print(f"Tanpa academic match    : {len(no_academic_match_1)} kandidat")

display(
    cleaned_df[
        [
            "nama_lengkap",
            "jurusan_angkatan",
            "pilihan_divisi_pertama",
            "matched_skills_divisi_pertama",
            "matched_academic_divisi_pertama",
            "matched_keywords_divisi_pertama"
        ]
    ].head(20)
)

=== CEK MATCHING KOSONG UNTUK PILIHAN DIVISI PERTAMA ===
Tanpa skill match       : 6 kandidat
Tanpa keyword match     : 15 kandidat
Tanpa academic match    : 64 kandidat


,nama_lengkap,jurusan_angkatan,pilihan_divisi_pertama,matched_skills_divisi_pertama,matched_academic_divisi_pertama,matched_keywords_divisi_pertama
0,Tomi Wijaya,Manajemen - 2022,HR Organizational Development,organizational development,Manajemen,organizational development
1,Regina Meidika,Hukum - 2026,HR Organizational Development,Google Workspace; komunikasi; koordinasi; Micr...,Hukum,organizational development; pengelolaan dokume...
2,Angga Ian Saputra,Manajemen - 2021,HR Talent Acquisition,Google Workspace; komunikasi; Microsoft Office...,Manajemen,proses rekrutmen; talent acquisition
3,Malesta Witaka Kussumasari,Akuntansi - 2024,Data Analyst,analisis data; pengolahan data,Akuntansi,analisis data; menganalisis data; pengolahan data
4,Ine Safitri Akes Parema,Psikologi - 2023,HR Talent Acquisition,communication; employee engagement; employer b...,Psikologi,employee engagement; employer branding; offeri...
5,Muhammad Rafli Choirul Umam,Pendidikan Bisnis - 2020,Data Analyst,,,
6,Nicholas Noverhino Ama Payong,Teknik Informatika - 2024,Data Analyst,analisis data; dashboard; data analysis; datab...,Informatika; Teknik Informatika,analisis data; dashboard; data analysis; datab...
7,Nindya Meiannur Shahnaz Purnomo,Statistika - 2025,Data Analyst,analytical thinking; attention to detail; crit...,Statistika,data analysis; data visualization; database; r...
8,Suhairi Ibnu Abdillah,Ekonomi Pembangunan-2022,Graphic Design,graphic design,,graphic design
9,Matthew Arthur Christopher Nathanael Hutapea,Sastra Inggris - 2021,Customer Services,customer service; komunikasi; problem solving,,customer service; ramah


In [ ]:
# UPDATE FLAG KE SCREENING OUTPUT


if "screening_output_df" in globals():
    screening_output_df["Flag"] = cleaned_df["flag"]
    print(" Flag pada screening_output_df berhasil diperbarui.")
else:
    print(" screening_output_df belum ditemukan. Nanti akan dibuat atau diperbarui di bagian export.")

 Flag pada screening_output_df berhasil diperbarui.


In [ ]:
# RINGKASAN BAGIAN 6


print("=== RINGKASAN BAGIAN 6 ===")
print(f"Jumlah kandidat diproses: {len(cleaned_df)}")

new_columns = [
    "matched_skills_divisi_pertama",
    "matched_academic_divisi_pertama",
    "matched_keywords_divisi_pertama",
    "skill_score_divisi_pertama",
    "academic_score_divisi_pertama",
    "relatedness_score_divisi_pertama",
    "matched_skills_divisi_kedua",
    "matched_academic_divisi_kedua",
    "matched_keywords_divisi_kedua",
    "skill_score_divisi_kedua",
    "academic_score_divisi_kedua",
    "relatedness_score_divisi_kedua"
]

print("\nKolom baru yang ditambahkan:")
for col in new_columns:
    print(f"- {col}")

print("\n Bagian 6 selesai.")
print("Lanjut ke Bagian 7 — Scoring Akhir dan Rekomendasi.")

=== RINGKASAN BAGIAN 6 ===
Jumlah kandidat diproses: 271

Kolom baru yang ditambahkan:
- matched_skills_divisi_pertama
- matched_academic_divisi_pertama
- matched_keywords_divisi_pertama
- skill_score_divisi_pertama
- academic_score_divisi_pertama
- relatedness_score_divisi_pertama
- matched_skills_divisi_kedua
- matched_academic_divisi_kedua
- matched_keywords_divisi_kedua
- skill_score_divisi_kedua
- academic_score_divisi_kedua
- relatedness_score_divisi_kedua

 Bagian 6 selesai.
Lanjut ke Bagian 7 — Scoring Akhir dan Rekomendasi.


In [ ]:
# BAGIAN 7 — SCORING AKHIR DAN REKOMENDASI


import re
import pandas as pd
import numpy as np

In [ ]:
# VALIDASI DEPENDENCY BAGIAN SEBELUMNYA


required_columns_for_scoring = [
    "nama_lengkap",
    "pilihan_divisi_pertama",
    "pilihan_divisi_kedua",
    "candidate_matching_text",
    "relatedness_score_divisi_pertama",
    "skill_score_divisi_pertama",
    "academic_score_divisi_pertama",
    "relatedness_score_divisi_kedua",
    "skill_score_divisi_kedua",
    "academic_score_divisi_kedua",
    "flag",
    "status_administrasi"
]

missing_columns = [
    col for col in required_columns_for_scoring
    if col not in cleaned_df.columns
]

if len(missing_columns) > 0:
    raise ValueError(
        "Kolom berikut belum tersedia. Pastikan Bagian 1-6 sudah dijalankan: "
        + ", ".join(missing_columns)
    )

print(" Dependency Bagian 7 sudah siap.")

 Dependency Bagian 7 sudah siap.


In [ ]:
# KONFIGURASI BOBOT DAN THRESHOLD


# Bobot final otomatis.
# Portofolio, Grammar, dan CV Structure tidak dimasukkan ke skor otomatis.
FINAL_SCORE_WEIGHTS = {
    "roles": 0.25,
    "relatedness": 0.35,
    "skill": 0.20,
    "academic": 0.20
}

# Threshold rekomendasi status
LOLOS_THRESHOLD = 70
DIPERTIMBANGKAN_THRESHOLD = 60

# Jika True, beberapa flag kritis dapat membuat rekomendasi status langsung Tidak Lolos.
# Bisa diubah menjadi False jika HR TA ingin status hanya berdasarkan skor.
APPLY_CRITICAL_FLAG_OVERRIDE = True

CRITICAL_FLAGS_FOR_NOT_LOLOS = [
    "MISSING_REQUIRED_DATA",
    "MISSING_CV",
    "INVALID_CV_LINK",
    "CV_DOWNLOAD_FAILED",
    "CV_EXTRACTION_FAILED",
    "MISSING_ADMIN_PROOF"
]

print("=== KONFIGURASI SCORING ===")
print("Bobot:")
for key, value in FINAL_SCORE_WEIGHTS.items():
    print(f"- {key}: {value}")

print()
print(f"Threshold Lolos              : >= {LOLOS_THRESHOLD}%")
print(f"Threshold Dipertimbangkan    : >= {DIPERTIMBANGKAN_THRESHOLD}% dan < {LOLOS_THRESHOLD}%")
print(f"Critical Flag Override       : {APPLY_CRITICAL_FLAG_OVERRIDE}")

=== KONFIGURASI SCORING ===
Bobot:
- roles: 0.25
- relatedness: 0.35
- skill: 0.2
- academic: 0.2

Threshold Lolos              : >= 70%
Threshold Dipertimbangkan    : >= 60% dan < 70%
Critical Flag Override       : True


In [ ]:
# ROLES & RESPONSIBILITY SCORING


RESPONSIBILITY_KEYWORDS = [
    # Bahasa Indonesia
    "mengelola", "membuat", "menyusun", "menganalisis", "mengembangkan",
    "mengkoordinasikan", "mengorganisir", "memimpin", "melakukan",
    "menangani", "bertanggung jawab", "membantu", "menyiapkan",
    "merancang", "menjalankan", "mengimplementasikan", "menyelesaikan",
    "mengawasi", "memantau", "mengevaluasi",

    # English
    "managed", "created", "prepared", "analyzed", "developed",
    "coordinated", "organized", "led", "conducted", "handled",
    "responsible for", "supported", "designed", "executed",
    "implemented", "completed", "monitored", "evaluated",
    "maintained", "delivered"
]

ACHIEVEMENT_KEYWORDS = [
    # Bahasa Indonesia
    "berhasil", "meningkatkan", "mencapai", "mengurangi", "memperbaiki",
    "mengoptimalkan", "memenangkan", "menghasilkan", "berdampak",
    "tercapai", "melampaui", "menyukseskan",

    # English
    "achieved", "improved", "increased", "reduced", "optimized",
    "won", "generated", "delivered", "exceeded", "successfully",
    "impact", "resulted in", "accomplished"
]

EXPERIENCE_SECTION_KEYWORDS = [
    "experience", "work experience", "pengalaman", "pengalaman kerja",
    "organization", "organisasi", "project", "proyek", "internship",
    "magang", "volunteer", "committee", "kepanitiaan", "leadership"
]


def has_quantitative_signal(text):
    """
    Mengecek indikasi data kuantitatif.
    Contoh:
    - 30%
    - 100 peserta
    - 20 candidates
    - 3 projects
    """
    text_norm = normalize_for_matching(text)

    quantitative_patterns = [
        r"\b\d+\s*%",
        r"\b\d+\s*percent\b",
        r"\b\d+\s*persen\b",
        r"\b\d+\+",
        r"\b\d+\s*(orang|peserta|kandidat|candidates|participants|followers|anggota|members|project|projects|event|events|konten|contents)\b"
    ]

    for pattern in quantitative_patterns:
        if re.search(pattern, text_norm):
            return True

    return False


def count_keyword_matches(text, keywords):
    """
    Menghitung jumlah keyword yang muncul di teks.
    """
    count = 0
    matched = []

    for keyword in keywords:
        if contains_keyword(text, keyword):
            count += 1
            matched.append(keyword)

    return count, matched


def calculate_roles_score(row):
    """
    Menghitung Roles & Responsibility Score dalam skala 1-5.

    Skor:
    1 = hampir tidak ada pengalaman/tanggung jawab
    2 = pengalaman ada tapi masih umum
    3 = tanggung jawab cukup spesifik
    4 = ada achievement
    5 = achievement + data/angka terukur
    """

    cv_text = row.get("cv_text", "")
    candidate_text = row.get("candidate_matching_text", "")

    # Prioritaskan teks CV. Jika tidak ada, pakai candidate_matching_text.
    text = cv_text if not is_blank(cv_text) else candidate_text
    text_norm = normalize_for_matching(text)

    if is_blank(text_norm):
        return 1, "Teks CV tidak tersedia, sehingga Roles & Responsibility sulit dinilai."

    responsibility_count, matched_responsibility = count_keyword_matches(
        text_norm,
        RESPONSIBILITY_KEYWORDS
    )

    achievement_count, matched_achievement = count_keyword_matches(
        text_norm,
        ACHIEVEMENT_KEYWORDS
    )

    experience_count, matched_experience = count_keyword_matches(
        text_norm,
        EXPERIENCE_SECTION_KEYWORDS
    )

    has_quant = has_quantitative_signal(text_norm)

    # Rule scoring
    if experience_count == 0 and responsibility_count == 0:
        score = 1
        reason = "Tidak ditemukan indikasi pengalaman atau tanggung jawab yang jelas."

    elif responsibility_count <= 2 and achievement_count == 0:
        score = 2
        reason = "Ditemukan pengalaman/tanggung jawab, tetapi penjelasannya masih umum."

    elif responsibility_count >= 3 and achievement_count == 0:
        score = 3
        reason = "Ditemukan beberapa tanggung jawab yang cukup spesifik."

    elif achievement_count > 0 and not has_quant:
        score = 4
        reason = "Ditemukan indikasi achievement/pencapaian, tetapi belum ada data terukur yang jelas."

    elif achievement_count > 0 and has_quant:
        score = 5
        reason = "Ditemukan achievement/pencapaian yang didukung angka atau data terukur."

    else:
        score = 3
        reason = "Ditemukan informasi tanggung jawab yang cukup, tetapi belum kuat pada achievement."

    return score, reason

In [ ]:
# HITUNG ROLES SCORE UNTUK SEMUA KANDIDAT


roles_score_list = []
roles_reason_list = []

for idx, row in cleaned_df.iterrows():
    roles_score, roles_reason = calculate_roles_score(row)
    roles_score_list.append(roles_score)
    roles_reason_list.append(roles_reason)

cleaned_df["roles_score"] = roles_score_list
cleaned_df["roles_reason"] = roles_reason_list

print(" Roles & Responsibility Score selesai dihitung.")

display(
    cleaned_df[
        [
            "nama_lengkap",
            "roles_score",
            "roles_reason"
        ]
    ].head(10)
)

 Roles & Responsibility Score selesai dihitung.


,nama_lengkap,roles_score,roles_reason
0,Tomi Wijaya,2,"Ditemukan pengalaman/tanggung jawab, tetapi pe..."
1,Regina Meidika,3,Ditemukan beberapa tanggung jawab yang cukup s...
2,Angga Ian Saputra,4,"Ditemukan indikasi achievement/pencapaian, tet..."
3,Malesta Witaka Kussumasari,4,"Ditemukan indikasi achievement/pencapaian, tet..."
4,Ine Safitri Akes Parema,2,"Ditemukan pengalaman/tanggung jawab, tetapi pe..."
5,Muhammad Rafli Choirul Umam,2,"Ditemukan pengalaman/tanggung jawab, tetapi pe..."
6,Nicholas Noverhino Ama Payong,5,Ditemukan achievement/pencapaian yang didukung...
7,Nindya Meiannur Shahnaz Purnomo,5,Ditemukan achievement/pencapaian yang didukung...
8,Suhairi Ibnu Abdillah,4,"Ditemukan indikasi achievement/pencapaian, tet..."
9,Matthew Arthur Christopher Nathanael Hutapea,4,"Ditemukan indikasi achievement/pencapaian, tet..."


In [ ]:
# FUNGSI HITUNG SKOR AKHIR


def safe_score(value, default=1):
    """
    Memastikan score numerik dan berada pada rentang 1-5.
    """
    try:
        value = float(value)
    except:
        value = default

    if value < 1:
        value = 1
    elif value > 5:
        value = 5

    return value


def calculate_final_score_percent(
    roles_score,
    relatedness_score,
    skill_score,
    academic_score
):
    """
    Menghitung skor akhir dalam persentase 0-100.
    Komponen internal tetap skala 1-5.
    """

    roles_score = safe_score(roles_score)
    relatedness_score = safe_score(relatedness_score)
    skill_score = safe_score(skill_score)
    academic_score = safe_score(academic_score)

    weighted_score_1_to_5 = (
        roles_score * FINAL_SCORE_WEIGHTS["roles"]
        + relatedness_score * FINAL_SCORE_WEIGHTS["relatedness"]
        + skill_score * FINAL_SCORE_WEIGHTS["skill"]
        + academic_score * FINAL_SCORE_WEIGHTS["academic"]
    )

    final_percent = (weighted_score_1_to_5 / 5) * 100

    return round(final_percent, 1)

In [ ]:
# HITUNG SKOR DIVISI PERTAMA DAN KEDUA


skor_divisi_pertama_list = []
skor_divisi_kedua_list = []

for idx, row in cleaned_df.iterrows():

    # Divisi pertama
    skor_pertama = calculate_final_score_percent(
        roles_score=row.get("roles_score", 1),
        relatedness_score=row.get("relatedness_score_divisi_pertama", 1),
        skill_score=row.get("skill_score_divisi_pertama", 1),
        academic_score=row.get("academic_score_divisi_pertama", 1)
    )

    # Divisi kedua
    skor_kedua = calculate_final_score_percent(
        roles_score=row.get("roles_score", 1),
        relatedness_score=row.get("relatedness_score_divisi_kedua", 1),
        skill_score=row.get("skill_score_divisi_kedua", 1),
        academic_score=row.get("academic_score_divisi_kedua", 1)
    )

    skor_divisi_pertama_list.append(skor_pertama)
    skor_divisi_kedua_list.append(skor_kedua)

cleaned_df["skor_divisi_pertama"] = skor_divisi_pertama_list
cleaned_df["skor_divisi_kedua"] = skor_divisi_kedua_list

print(" Skor Divisi Pertama dan Skor Divisi Kedua berhasil dihitung dalam bentuk persentase.")

display(
    cleaned_df[
        [
            "nama_lengkap",
            "pilihan_divisi_pertama",
            "skor_divisi_pertama",
            "pilihan_divisi_kedua",
            "skor_divisi_kedua"
        ]
    ].head(10)
)

 Skor Divisi Pertama dan Skor Divisi Kedua berhasil dihitung dalam bentuk persentase.


,nama_lengkap,pilihan_divisi_pertama,skor_divisi_pertama,pilihan_divisi_kedua,skor_divisi_kedua
0,Tomi Wijaya,HR Organizational Development,48.0,Business Development,48.0
1,Regina Meidika,HR Organizational Development,72.0,HR Talent Acquisition,72.0
2,Angga Ian Saputra,HR Talent Acquisition,74.0,HR Talent Acquisition,74.0
3,Malesta Witaka Kussumasari,Data Analyst,65.0,Data Analyst,65.0
4,Ine Safitri Akes Parema,HR Talent Acquisition,85.0,HR Organizational Development,85.0
5,Muhammad Rafli Choirul Umam,Data Analyst,25.0,Data Analyst,25.0
6,Nicholas Noverhino Ama Payong,Data Analyst,100.0,Data Analyst,100.0
7,Nindya Meiannur Shahnaz Purnomo,Data Analyst,86.0,Business Development,70.0
8,Suhairi Ibnu Abdillah,Graphic Design,42.0,Content Creator,35.0
9,Matthew Arthur Christopher Nathanael Hutapea,Customer Services,50.0,Event Project,46.0


In [ ]:
# TENTUKAN REKOMENDASI DIVISI


def determine_recommended_division(row):
    skor_pertama = row.get("skor_divisi_pertama", 0)
    skor_kedua = row.get("skor_divisi_kedua", 0)

    divisi_pertama = row.get("pilihan_divisi_pertama", "")
    divisi_kedua = row.get("pilihan_divisi_kedua", "")

    if safe_score(skor_pertama, default=0) >= safe_score(skor_kedua, default=0):
        return divisi_pertama
    else:
        return divisi_kedua


def determine_best_score(row):
    skor_pertama = float(row.get("skor_divisi_pertama", 0))
    skor_kedua = float(row.get("skor_divisi_kedua", 0))
    return max(skor_pertama, skor_kedua)


cleaned_df["rekomendasi_divisi"] = cleaned_df.apply(
    determine_recommended_division,
    axis=1
)

cleaned_df["skor_terbaik"] = cleaned_df.apply(
    determine_best_score,
    axis=1
)

print(" Rekomendasi divisi berhasil dibuat.")

display(
    cleaned_df[
        [
            "nama_lengkap",
            "pilihan_divisi_pertama",
            "skor_divisi_pertama",
            "pilihan_divisi_kedua",
            "skor_divisi_kedua",
            "rekomendasi_divisi",
            "skor_terbaik"
        ]
    ].head(10)
)

 Rekomendasi divisi berhasil dibuat.


,nama_lengkap,pilihan_divisi_pertama,skor_divisi_pertama,pilihan_divisi_kedua,skor_divisi_kedua,rekomendasi_divisi,skor_terbaik
0,Tomi Wijaya,HR Organizational Development,48.0,Business Development,48.0,HR Organizational Development,48.0
1,Regina Meidika,HR Organizational Development,72.0,HR Talent Acquisition,72.0,HR Organizational Development,72.0
2,Angga Ian Saputra,HR Talent Acquisition,74.0,HR Talent Acquisition,74.0,HR Talent Acquisition,74.0
3,Malesta Witaka Kussumasari,Data Analyst,65.0,Data Analyst,65.0,Data Analyst,65.0
4,Ine Safitri Akes Parema,HR Talent Acquisition,85.0,HR Organizational Development,85.0,HR Talent Acquisition,85.0
5,Muhammad Rafli Choirul Umam,Data Analyst,25.0,Data Analyst,25.0,Data Analyst,25.0
6,Nicholas Noverhino Ama Payong,Data Analyst,100.0,Data Analyst,100.0,Data Analyst,100.0
7,Nindya Meiannur Shahnaz Purnomo,Data Analyst,86.0,Business Development,70.0,Data Analyst,86.0
8,Suhairi Ibnu Abdillah,Graphic Design,42.0,Content Creator,35.0,Graphic Design,42.0
9,Matthew Arthur Christopher Nathanael Hutapea,Customer Services,50.0,Event Project,46.0,Customer Services,50.0


In [ ]:
# TENTUKAN REKOMENDASI STATUS


def has_critical_flag(flag_text):
    flag_text = clean_text(flag_text)

    if flag_text == "":
        return False

    flags = [flag.strip() for flag in flag_text.split(";") if flag.strip()]

    for flag in flags:
        if flag in CRITICAL_FLAGS_FOR_NOT_LOLOS:
            return True

    return False


def determine_recommendation_status(row):
    skor_terbaik = float(row.get("skor_terbaik", 0))
    flag_text = row.get("flag", "")

    # Override jika ada flag kritis
    if APPLY_CRITICAL_FLAG_OVERRIDE and has_critical_flag(flag_text):
        return "Tidak Lolos"

    # Status berdasarkan skor
    if skor_terbaik >= LOLOS_THRESHOLD:
        return "Lolos"
    elif skor_terbaik >= DIPERTIMBANGKAN_THRESHOLD:
        return "Dipertimbangkan"
    else:
        return "Tidak Lolos"


cleaned_df["rekomendasi_status"] = cleaned_df.apply(
    determine_recommendation_status,
    axis=1
)

print(" Rekomendasi status berhasil dibuat.")

display(
    cleaned_df[
        [
            "nama_lengkap",
            "skor_terbaik",
            "rekomendasi_divisi",
            "rekomendasi_status",
            "flag"
        ]
    ].head(10)
)

 Rekomendasi status berhasil dibuat.


,nama_lengkap,skor_terbaik,rekomendasi_divisi,rekomendasi_status,flag
0,Tomi Wijaya,48.0,HR Organizational Development,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
1,Regina Meidika,72.0,HR Organizational Development,Lolos,
2,Angga Ian Saputra,74.0,HR Talent Acquisition,Lolos,
3,Malesta Witaka Kussumasari,65.0,Data Analyst,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
4,Ine Safitri Akes Parema,85.0,HR Talent Acquisition,Lolos,
5,Muhammad Rafli Choirul Umam,25.0,Data Analyst,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
6,Nicholas Noverhino Ama Payong,100.0,Data Analyst,Lolos,
7,Nindya Meiannur Shahnaz Purnomo,86.0,Data Analyst,Lolos,
8,Suhairi Ibnu Abdillah,42.0,Graphic Design,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
9,Matthew Arthur Christopher Nathanael Hutapea,50.0,Customer Services,Tidak Lolos,CV_DOWNLOAD_FAILED


In [ ]:
# BUAT ALASAN OTOMATIS


def get_component_scores_for_recommended_division(row):
    """
    Mengambil komponen skor sesuai divisi yang direkomendasikan.
    """
    rekomendasi_divisi = row.get("rekomendasi_divisi", "")
    divisi_pertama = row.get("pilihan_divisi_pertama", "")

    if clean_text(rekomendasi_divisi).lower() == clean_text(divisi_pertama).lower():
        return {
            "relatedness": row.get("relatedness_score_divisi_pertama", 1),
            "skill": row.get("skill_score_divisi_pertama", 1),
            "academic": row.get("academic_score_divisi_pertama", 1),
            "matched_skills": row.get("matched_skills_divisi_pertama", ""),
            "matched_academic": row.get("matched_academic_divisi_pertama", ""),
            "matched_keywords": row.get("matched_keywords_divisi_pertama", "")
        }
    else:
        return {
            "relatedness": row.get("relatedness_score_divisi_kedua", 1),
            "skill": row.get("skill_score_divisi_kedua", 1),
            "academic": row.get("academic_score_divisi_kedua", 1),
            "matched_skills": row.get("matched_skills_divisi_kedua", ""),
            "matched_academic": row.get("matched_academic_divisi_kedua", ""),
            "matched_keywords": row.get("matched_keywords_divisi_kedua", "")
        }


def build_auto_reason(row):
    nama = row.get("nama_lengkap", "")
    divisi_pertama = row.get("pilihan_divisi_pertama", "")
    divisi_kedua = row.get("pilihan_divisi_kedua", "")
    skor_pertama = row.get("skor_divisi_pertama", 0)
    skor_kedua = row.get("skor_divisi_kedua", 0)
    rekomendasi_divisi = row.get("rekomendasi_divisi", "")
    rekomendasi_status = row.get("rekomendasi_status", "")
    flag_text = row.get("flag", "")
    roles_score = row.get("roles_score", 1)
    roles_reason = row.get("roles_reason", "")

    component = get_component_scores_for_recommended_division(row)

    relatedness = component["relatedness"]
    skill = component["skill"]
    academic = component["academic"]
    matched_skills = component["matched_skills"]
    matched_academic = component["matched_academic"]
    matched_keywords = component["matched_keywords"]

    reason_parts = []

    reason_parts.append(
        f"Kandidat direkomendasikan untuk {rekomendasi_divisi} dengan skor terbaik {row.get('skor_terbaik', 0)}%."
    )

    reason_parts.append(
        f"Perbandingan skor: {divisi_pertama} {skor_pertama}%, {divisi_kedua} {skor_kedua}%."
    )

    reason_parts.append(
        f"Komponen utama: Roles {roles_score}/5, Relatedness {relatedness}/5, Skill {skill}/5, Academic {academic}/5."
    )

    if not is_blank(matched_skills):
        reason_parts.append(f"Skill terdeteksi: {matched_skills}.")

    if not is_blank(matched_academic):
        reason_parts.append(f"Jurusan relevan terdeteksi: {matched_academic}.")

    if not is_blank(matched_keywords):
        reason_parts.append(f"Keyword pengalaman relevan: {matched_keywords}.")

    if not is_blank(roles_reason):
        reason_parts.append(f"Catatan roles: {roles_reason}")

    if not is_blank(flag_text):
        reason_parts.append(f"Flag yang perlu dicek: {flag_text}.")

    reason_parts.append(f"Rekomendasi status sistem: {rekomendasi_status}.")

    return " ".join(reason_parts)


cleaned_df["alasan_otomatis"] = cleaned_df.apply(
    build_auto_reason,
    axis=1
)

print(" Alasan otomatis berhasil dibuat.")

display(
    cleaned_df[
        [
            "nama_lengkap",
            "rekomendasi_divisi",
            "rekomendasi_status",
            "alasan_otomatis"
        ]
    ].head(5)
)

 Alasan otomatis berhasil dibuat.


,nama_lengkap,rekomendasi_divisi,rekomendasi_status,alasan_otomatis
0,Tomi Wijaya,HR Organizational Development,Tidak Lolos,Kandidat direkomendasikan untuk HR Organizatio...
1,Regina Meidika,HR Organizational Development,Lolos,Kandidat direkomendasikan untuk HR Organizatio...
2,Angga Ian Saputra,HR Talent Acquisition,Lolos,Kandidat direkomendasikan untuk HR Talent Acqu...
3,Malesta Witaka Kussumasari,Data Analyst,Tidak Lolos,Kandidat direkomendasikan untuk Data Analyst d...
4,Ine Safitri Akes Parema,HR Talent Acquisition,Lolos,Kandidat direkomendasikan untuk HR Talent Acqu...


In [ ]:
# UPDATE SCREENING OUTPUT DATAFRAME


# Jika screening_output_df belum ada, buat ulang dari cleaned_df
if "screening_output_df" not in globals():
    print(" screening_output_df belum ditemukan. Membuat ulang berdasarkan cleaned_df.")

    output_columns = [
        "No",
        "Nama Lengkap",
        "Nomor WhatsApp",
        "Email",
        "Pendidikan",
        "Jurusan dan Angkatan",
        "Asal Instansi",
        "Instagram",
        "Pilihan Divisi Pertama",
        "Pilihan Posisi Pertama",
        "Pilihan Divisi Kedua",
        "Pilihan Posisi Kedua",
        "Link CV",
        "Link Portofolio",
        "Status Administrasi",
        "Flag",
        "Skor Divisi Pertama",
        "Skor Divisi Kedua",
        "Rekomendasi Divisi",
        "Rekomendasi Status",
        "Alasan Otomatis",
        "Catatan Reviewer",
        "Status Final Reviewer",
        "Alasan Final Reviewer"
    ]

    screening_output_df = pd.DataFrame()
    screening_output_df["No"] = cleaned_df["no"]
    screening_output_df["Nama Lengkap"] = cleaned_df["nama_lengkap"]
    screening_output_df["Nomor WhatsApp"] = cleaned_df["nomor_whatsapp"]
    screening_output_df["Email"] = cleaned_df["email"]
    screening_output_df["Pendidikan"] = cleaned_df["pendidikan"]
    screening_output_df["Jurusan dan Angkatan"] = cleaned_df["jurusan_angkatan"]
    screening_output_df["Asal Instansi"] = cleaned_df["asal_instansi"]
    screening_output_df["Instagram"] = cleaned_df["instagram"]
    screening_output_df["Pilihan Divisi Pertama"] = cleaned_df["pilihan_divisi_pertama"]
    screening_output_df["Pilihan Posisi Pertama"] = cleaned_df["pilihan_posisi_pertama"]
    screening_output_df["Pilihan Divisi Kedua"] = cleaned_df["pilihan_divisi_kedua"]
    screening_output_df["Pilihan Posisi Kedua"] = cleaned_df["pilihan_posisi_kedua"]
    screening_output_df["Link CV"] = cleaned_df["link_cv"]
    screening_output_df["Link Portofolio"] = cleaned_df["link_portofolio"]
    screening_output_df["Catatan Reviewer"] = ""
    screening_output_df["Status Final Reviewer"] = ""
    screening_output_df["Alasan Final Reviewer"] = ""

# Update kolom hasil sistem
screening_output_df["Status Administrasi"] = cleaned_df["status_administrasi"]
screening_output_df["Flag"] = cleaned_df["flag"]

# Format skor sebagai angka persentase, bukan string.
# Nanti di Excel bisa diformat sebagai 0.0%.
screening_output_df["Skor Divisi Pertama"] = cleaned_df["skor_divisi_pertama"]
screening_output_df["Skor Divisi Kedua"] = cleaned_df["skor_divisi_kedua"]

screening_output_df["Rekomendasi Divisi"] = cleaned_df["rekomendasi_divisi"]
screening_output_df["Rekomendasi Status"] = cleaned_df["rekomendasi_status"]
screening_output_df["Alasan Otomatis"] = cleaned_df["alasan_otomatis"]

# Pastikan urutan kolom sesuai template
output_columns = [
    "No",
    "Nama Lengkap",
    "Nomor WhatsApp",
    "Email",
    "Pendidikan",
    "Jurusan dan Angkatan",
    "Asal Instansi",
    "Instagram",
    "Pilihan Divisi Pertama",
    "Pilihan Posisi Pertama",
    "Pilihan Divisi Kedua",
    "Pilihan Posisi Kedua",
    "Link CV",
    "Link Portofolio",
    "Status Administrasi",
    "Flag",
    "Skor Divisi Pertama",
    "Skor Divisi Kedua",
    "Rekomendasi Divisi",
    "Rekomendasi Status",
    "Alasan Otomatis",
    "Catatan Reviewer",
    "Status Final Reviewer",
    "Alasan Final Reviewer"
]

screening_output_df = screening_output_df[output_columns]

print(" screening_output_df berhasil diperbarui dengan hasil scoring akhir.")

display(screening_output_df.head(10))

 screening_output_df berhasil diperbarui dengan hasil scoring akhir.


,No,Nama Lengkap,Nomor WhatsApp,Email,Pendidikan,Jurusan dan Angkatan,Asal Instansi,Instagram,Pilihan Divisi Pertama,Pilihan Posisi Pertama,...,Status Administrasi,Flag,Skor Divisi Pertama,Skor Divisi Kedua,Rekomendasi Divisi,Rekomendasi Status,Alasan Otomatis,Catatan Reviewer,Status Final Reviewer,Alasan Final Reviewer
0,1,Tomi Wijaya,wa.me/6285388990284,Tommywijaya315@gmail.com,Mahasiswa,Manajemen - 2022,Universitas Bhayangkara Jakarta Raya,https://www.instagram.com/tommywijy/,HR Organizational Development,Staff,...,Perlu Dicek,CV_LINK_IS_FOLDER; INVALID_CV_LINK,48.0,48.0,HR Organizational Development,Tidak Lolos,Kandidat direkomendasikan untuk HR Organizatio...,,,
1,2,Regina Meidika,081316537670,reginameidika30@gmail.com,Fresh Graduate,Hukum - 2026,Universitas Lampung,https://www.instagram.com/reginameii_?igsh=MWc...,HR Organizational Development,Staff,...,Administrasi Lengkap,,72.0,72.0,HR Organizational Development,Lolos,Kandidat direkomendasikan untuk HR Organizatio...,,,
2,3,Angga Ian Saputra,wa.me/6287804032180,anggaiansaputra06@gmail.com,Fresh Graduate,Manajemen - 2021,Universitas Bhayangkara Jakarta Raya,instagram.com/sanggaian,HR Talent Acquisition,Staff,...,Administrasi Lengkap,,74.0,74.0,HR Talent Acquisition,Lolos,Kandidat direkomendasikan untuk HR Talent Acqu...,,,
3,4,Malesta Witaka Kussumasari,wa.me/62895401664763,malestawitaka@gmail.com,Mahasiswa,Akuntansi - 2024,UPN Veteran Yogyakarta,instagram.com/maalesta,Data Analyst,Staff,...,Perlu Dicek,CV_LINK_IS_FOLDER; INVALID_CV_LINK,65.0,65.0,Data Analyst,Tidak Lolos,Kandidat direkomendasikan untuk Data Analyst d...,,,
4,5,Ine Safitri Akes Parema,wa.me/6282314686358,ineysafitri@gmail.com,Mahasiswa,Psikologi - 2023,Universitas Negeri Semarang,instagram.com/h0neyy_neyy,HR Talent Acquisition,Staff,...,Administrasi Lengkap,,85.0,85.0,HR Talent Acquisition,Lolos,Kandidat direkomendasikan untuk HR Talent Acqu...,,,
5,6,Muhammad Rafli Choirul Umam,wa.me/628979934206,mraflichoirulumam@gmail.com,Fresh Graduate,Pendidikan Bisnis - 2020,Universitas Negeri Jakarta,instagram.com/choirul_umam02,Data Analyst,Staff,...,Perlu Dicek,CV_LINK_IS_FOLDER; INVALID_CV_LINK,25.0,25.0,Data Analyst,Tidak Lolos,Kandidat direkomendasikan untuk Data Analyst d...,,,
6,7,Nicholas Noverhino Ama Payong,wa.me/6285245586655,nicholasnoverhino@gmail.com,Mahasiswa,Teknik Informatika - 2024,Universitas Negeri Semarang,instagram.com/nichoonvrhn_,Data Analyst,Staff,...,Administrasi Lengkap,,100.0,100.0,Data Analyst,Lolos,Kandidat direkomendasikan untuk Data Analyst d...,,,
7,8,Nindya Meiannur Shahnaz Purnomo,wa.me/6285251770404,nindyameiannur@gmail.com,Fresh Graduate,Statistika - 2025,Universitas Brawijaya,instagram.com/conutmatcha_,Data Analyst,Staff,...,Administrasi Lengkap,,86.0,70.0,Data Analyst,Lolos,Kandidat direkomendasikan untuk Data Analyst d...,,,
8,9,Suhairi Ibnu Abdillah,087858547217,suhairiibnuabdillah@gmail.com,Mahasiswa,Ekonomi Pembangunan-2022,Universitas Bojonegoro,abdillah.ib,Graphic Design,Staff,...,Perlu Dicek,CV_LINK_IS_FOLDER; INVALID_CV_LINK,42.0,35.0,Graphic Design,Tidak Lolos,Kandidat direkomendasikan untuk Graphic Design...,,,
9,10,Matthew Arthur Christopher Nathanael Hutapea,628118789277,Matthewarthurhutapea@gmail.com,Fresh Graduate,Sastra Inggris - 2021,Universitas Kristen Indonesia,https://www.instagram.com/arthr.smile?igsh=Mjk...,Customer Services,Staff,...,Perlu Dicek,CV_DOWNLOAD_FAILED,50.0,46.0,Customer Services,Tidak Lolos,Kandidat direkomendasikan untuk Customer Servi...,,,


In [ ]:
# RINGKASAN SKOR AKHIR


print("=== RINGKASAN SKOR DIVISI PERTAMA ===")
display(cleaned_df["skor_divisi_pertama"].describe())

print("=== RINGKASAN SKOR DIVISI KEDUA ===")
display(cleaned_df["skor_divisi_kedua"].describe())

print("=== RINGKASAN REKOMENDASI STATUS ===")
display(cleaned_df["rekomendasi_status"].value_counts(dropna=False))

print("=== RINGKASAN REKOMENDASI DIVISI ===")
display(cleaned_df["rekomendasi_divisi"].value_counts(dropna=False))

=== RINGKASAN SKOR DIVISI PERTAMA ===


,skor_divisi_pertama
count,271.000000
mean,72.704797
std,18.579597
min,20.000000
25%,61.000000
50%,74.000000
75%,88.000000
max,100.000000


=== RINGKASAN SKOR DIVISI KEDUA ===


,skor_divisi_kedua
count,271.000000
mean,67.745387
std,17.794031
min,20.000000
25%,57.000000
50%,69.000000
75%,81.000000
max,100.000000


=== RINGKASAN REKOMENDASI STATUS ===


,count
rekomendasi_status,
Lolos,153
Tidak Lolos,90
Dipertimbangkan,28


=== RINGKASAN REKOMENDASI DIVISI ===


,count
rekomendasi_divisi,
Data Analyst,88
HR Organizational Development,37
HR Talent Acquisition,37
Event Project,34
Business Development,26
Customer Services,13
Graphic Design,13
CEO Staff,9
Social Media Specialist,6


In [ ]:
# PREVIEW HASIL BERDASARKAN STATUS


print("=== PREVIEW KANDIDAT LOLOS ===")
display(
    cleaned_df[
        cleaned_df["rekomendasi_status"] == "Lolos"
    ][
        [
            "nama_lengkap",
            "rekomendasi_divisi",
            "skor_terbaik",
            "rekomendasi_status",
            "flag"
        ]
    ].head(10)
)

print("=== PREVIEW KANDIDAT DIPERTIMBANGKAN ===")
display(
    cleaned_df[
        cleaned_df["rekomendasi_status"] == "Dipertimbangkan"
    ][
        [
            "nama_lengkap",
            "rekomendasi_divisi",
            "skor_terbaik",
            "rekomendasi_status",
            "flag"
        ]
    ].head(10)
)

print("=== PREVIEW KANDIDAT TIDAK LOLOS ===")
display(
    cleaned_df[
        cleaned_df["rekomendasi_status"] == "Tidak Lolos"
    ][
        [
            "nama_lengkap",
            "rekomendasi_divisi",
            "skor_terbaik",
            "rekomendasi_status",
            "flag"
        ]
    ].head(10)
)

=== PREVIEW KANDIDAT LOLOS ===


,nama_lengkap,rekomendasi_divisi,skor_terbaik,rekomendasi_status,flag
1,Regina Meidika,HR Organizational Development,72.0,Lolos,
2,Angga Ian Saputra,HR Talent Acquisition,74.0,Lolos,
4,Ine Safitri Akes Parema,HR Talent Acquisition,85.0,Lolos,
6,Nicholas Noverhino Ama Payong,Data Analyst,100.0,Lolos,
7,Nindya Meiannur Shahnaz Purnomo,Data Analyst,86.0,Lolos,
12,Ahmad Rahmatullah,Data Analyst,100.0,Lolos,
15,Gabriella Elroi Stephanie Silalahi,Event Project,71.0,Lolos,
17,Sunandar,CEO Staff,95.0,Lolos,
18,Anomie Parisurya,Customer Services,81.0,Lolos,
20,Difa Leroy Gladion,Data Analyst,74.0,Lolos,


=== PREVIEW KANDIDAT DIPERTIMBANGKAN ===


,nama_lengkap,rekomendasi_divisi,skor_terbaik,rekomendasi_status,flag
11,JOVAN ALDHY MULIAWAN,Data Analyst,64.0,Dipertimbangkan,
24,Siti Muliyani,Event Project,69.0,Dipertimbangkan,
43,Yasir Ibnu Basuni,Data Analyst,67.0,Dipertimbangkan,
44,Louis Oktovianus,Data Analyst,67.0,Dipertimbangkan,
46,Ananda Aurelia Anindia Putri,HR Organizational Development,63.0,Dipertimbangkan,
48,TIARA AISYAH RAMADHANI,HR Organizational Development,64.0,Dipertimbangkan,
64,Bening Kirani,Data Analyst,65.0,Dipertimbangkan,
89,Pradipta Al Ghazali,Social Media Specialist,64.0,Dipertimbangkan,
91,Amalia Ramadhani firdausiyah,Customer Services,60.0,Dipertimbangkan,
135,Riska Kodansyah,HR Talent Acquisition,69.0,Dipertimbangkan,


=== PREVIEW KANDIDAT TIDAK LOLOS ===


,nama_lengkap,rekomendasi_divisi,skor_terbaik,rekomendasi_status,flag
0,Tomi Wijaya,HR Organizational Development,48.0,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
3,Malesta Witaka Kussumasari,Data Analyst,65.0,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
5,Muhammad Rafli Choirul Umam,Data Analyst,25.0,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
8,Suhairi Ibnu Abdillah,Graphic Design,42.0,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
9,Matthew Arthur Christopher Nathanael Hutapea,Customer Services,50.0,Tidak Lolos,CV_DOWNLOAD_FAILED
10,Rachel Christine Meinauli Sitorus,Event Project,44.0,Tidak Lolos,
13,Zacki Ferdinansyah,Data Analyst,36.0,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
14,Dwi Musalim Mudrik,Business Development,44.0,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
16,DITA DWI RIYANI,Business Development,80.0,Tidak Lolos,CV_LINK_IS_FOLDER; INVALID_CV_LINK
19,Amanda Alvina,Customer Services,55.0,Tidak Lolos,


In [ ]:
# RINGKASAN BAGIAN 7


print("=== RINGKASAN BAGIAN 7 ===")
print(f"Jumlah kandidat diproses: {len(cleaned_df)}")
print(f"Threshold Lolos         : >= {LOLOS_THRESHOLD}%")
print(f"Threshold Dipertimbangkan: >= {DIPERTIMBANGKAN_THRESHOLD}% dan < {LOLOS_THRESHOLD}%")

print("\nKolom scoring final yang ditambahkan:")
final_scoring_columns = [
    "roles_score",
    "roles_reason",
    "skor_divisi_pertama",
    "skor_divisi_kedua",
    "skor_terbaik",
    "rekomendasi_divisi",
    "rekomendasi_status",
    "alasan_otomatis"
]

for col in final_scoring_columns:
    print(f"- {col}")

print("\n Bagian 7 selesai.")
print("Lanjut ke Bagian 8 — Export Output ke Excel.")

=== RINGKASAN BAGIAN 7 ===
Jumlah kandidat diproses: 271
Threshold Lolos         : >= 70%
Threshold Dipertimbangkan: >= 60% dan < 70%

Kolom scoring final yang ditambahkan:
- roles_score
- roles_reason
- skor_divisi_pertama
- skor_divisi_kedua
- skor_terbaik
- rekomendasi_divisi
- rekomendasi_status
- alasan_otomatis

 Bagian 7 selesai.
Lanjut ke Bagian 8 — Export Output ke Excel.


In [ ]:
# BAGIAN 8 — EXPORT OUTPUT KE EXCEL


!pip install -q openpyxl

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime

In [ ]:
# VALIDASI DATAFRAME OUTPUT


if "screening_output_df" not in globals():
    raise ValueError("screening_output_df belum tersedia. Jalankan Bagian 7 terlebih dahulu.")

if screening_output_df.empty:
    raise ValueError("screening_output_df kosong. Tidak ada data untuk diexport.")

print(" screening_output_df tersedia.")
print(f"Jumlah baris: {len(screening_output_df)}")
print(f"Jumlah kolom: {screening_output_df.shape[1]}")

 screening_output_df tersedia.
Jumlah baris: 271
Jumlah kolom: 24


In [ ]:
# SET NAMA FILE OUTPUT

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_FILE = OUTPUT_DIR / f"hasil_screening_{timestamp}.xlsx"

print(f"Output akan disimpan ke:")
print(OUTPUT_FILE)


Output akan disimpan ke:
/content/drive/Shareddrives/CV Screening Automation/02_output/hasil_screening_20260722_122029.xlsx


In [ ]:
# EXPORT DATAFRAME KE EXCEL


with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    screening_output_df.to_excel(
        writer,
        sheet_name="hasil_screening",
        index=False
    )

print(" Data berhasil diexport ke Excel.")

 Data berhasil diexport ke Excel.


In [ ]:
# FORMAT SHEET HASIL_SCREENING


wb = load_workbook(OUTPUT_FILE)
ws = wb["hasil_screening"]

# Warna dan style
header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
thin_border = Border(
    bottom=Side(style="thin", color="D9D9D9")
)

# Freeze pane
ws.freeze_panes = "A2"

# Auto filter
ws.auto_filter.ref = ws.dimensions

# Format header
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True
    )

# Format semua cell
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True
        )
        cell.border = thin_border

# Set lebar kolom
column_widths = {
    "A": 6,    # No
    "B": 28,   # Nama Lengkap
    "C": 20,   # Nomor WhatsApp
    "D": 30,   # Email
    "E": 18,   # Pendidikan
    "F": 30,   # Jurusan dan Angkatan
    "G": 30,   # Asal Instansi
    "H": 25,   # Instagram
    "I": 26,   # Pilihan Divisi Pertama
    "J": 22,   # Pilihan Posisi Pertama
    "K": 26,   # Pilihan Divisi Kedua
    "L": 22,   # Pilihan Posisi Kedua
    "M": 35,   # Link CV
    "N": 35,   # Link Portofolio
    "O": 24,   # Status Administrasi
    "P": 35,   # Flag
    "Q": 18,   # Skor Divisi Pertama
    "R": 18,   # Skor Divisi Kedua
    "S": 26,   # Rekomendasi Divisi
    "T": 22,   # Rekomendasi Status
    "U": 60,   # Alasan Otomatis
    "V": 45,   # Catatan Reviewer
    "W": 24,   # Status Final Reviewer
    "X": 45,   # Alasan Final Reviewer
}

for col_letter, width in column_widths.items():
    ws.column_dimensions[col_letter].width = width

# Format skor sebagai persentase tampilan.
# Karena nilai kita sudah 0-100, formatnya pakai angka biasa dengan simbol %
score_columns = ["Q", "R"]

for col in score_columns:
    for row in range(2, ws.max_row + 1):
        ws[f"{col}{row}"].number_format = '0.0"%"'

# Simpan
wb.save(OUTPUT_FILE)

print(" Formatting Excel selesai.")

 Formatting Excel selesai.


In [ ]:
# PREVIEW OUTPUT


print("=== EXPORT SELESAI ===")
print(f"File hasil screening berhasil dibuat:")
print(OUTPUT_FILE)

=== EXPORT SELESAI ===
File hasil screening berhasil dibuat:
/content/drive/Shareddrives/CV Screening Automation/02_output/hasil_screening_20260722_122029.xlsx
